In [ ]:
!pip -q install spacy transformers torch
!python -m spacy download el_core_news_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 34.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('el_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import re
import string
import pandas as pd
from sklearn.metrics import classification_report
import spacy
from transformers import pipeline

In [ ]:
df = pd.read_pickle("/content/drive/MyDrive/Archimedes_Anonymization/df.pkl")
df.head()

,file,raw_text,synthetic_text,synthetic_map,clean_text,gold_spans,template_text,free_text,gold_spans_template,gold_spans_free_text
0,Ασθ1.docx,##############################################...,##############################################...,"{'@74930277': {'type': 'patient_id', 'value': ...",##############################################...,"[(326, 334, patient_id), (403, 416, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ – ΠΟΡΕΙΑ ΝΟΣΟΥ\nΑτομι...,"[(326, 334, patient_id), (403, 416, staff_name...","[(298, 321, hospital_name), (366, 389, hospita..."
1,Ασθ2.docx,##############################################...,##############################################...,"{'@745625100': {'type': 'patient_id', 'value':...",##############################################...,"[(325, 333, patient_id), (414, 441, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\n\nΠρογραμματισμένη ε...,"[(325, 333, patient_id), (414, 441, staff_name...","[(60, 90, hospital_name), (2897, 2927, hospita..."
2,Ασθ3.docx,##############################################...,##############################################...,"{'@84726155': {'type': 'patient_id', 'value': ...",##############################################...,"[(326, 334, patient_id), (415, 434, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\nΑιτία εισόδου: Ασθεν...,"[(326, 334, patient_id), (415, 434, staff_name...","[(792, 822, hospital_name)]"
3,Ασθ4.docx,##############################################...,##############################################...,"{'@9832492': {'type': 'patient_id', 'value': '...",##############################################...,"[(325, 333, patient_id), (414, 427, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\nΑΙΤΙΑ ΕΙΣΟΔΟΥ:\nΟΠΟ-...,"[(325, 333, patient_id), (414, 427, staff_name...","[(953, 965, hospital_name)]"
4,Ασθ5.docx,##############################################...,##############################################...,"{'@0912483': {'type': 'patient_id', 'value': '...",##############################################...,"[(326, 334, patient_id), (415, 436, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\nΑιτία εισόδου :Χειρο...,"[(326, 334, patient_id), (415, 436, staff_name...",[]


TEMPLATE AWARE RULES

In [ ]:
#Τα labels στο template που πρεπει να κρυψουμε (π.χ. Όνομα, Επώνυμο, Διεύθυνση, Τηλέφωνο)
#μπορει να ειναι στην ιδια γραμμη ή στην επομενη

FIELD_SAME_LINE = [
    ("patient_id",    re.compile(r"(Αρ\.\s*Μητρ\.\s*Ασθ\.\s*:\s*)([^\n\r]+)")),
    ("staff_name",    re.compile(r"(Διευθυντής\s*:\s*)([^\n\r]+)")),
    ("last_name",     re.compile(r"(Επώνυμο\s*:\s*)([^\n\r]+)")),
    ("first_name",    re.compile(r"(Όνομα\s*:\s*)([^\n\r]+)")),
    ("address",       re.compile(r"(Διεύθυνση\s*:\s*)([^\n\r]+)")),
    ("postal_city",   re.compile(r"(Τ\.\s*Κ\.\s*–\s*Πόλη\s*:\s*)([^\n\r]+)")),
    ("phone",         re.compile(r"(Τηλέφωνο\s*:\s*)([^\n\r]+)")),
]

FIELD_NEXT_LINE = [
    ("patient_id",    re.compile(r"(Αρ\.\s*Μητρ\.\s*Ασθ\.\s*:\s*\n)([^\n\r]+)")),
    ("staff_name",    re.compile(r"(Διευθυντής\s*:\s*\n)([^\n\r]+)")),
    ("last_name",     re.compile(r"(Επώνυμο\s*:\s*\n)([^\n\r]+)")),
    ("first_name",    re.compile(r"(Όνομα\s*:\s*\n)([^\n\r]+)")),
    ("address",       re.compile(r"(Διεύθυνση\s*:\s*\n)([^\n\r]+)")),
    ("postal_city",   re.compile(r"(Τ\.\s*Κ\.\s*–\s*Πόλη\s*:\s*\n)([^\n\r]+)")),
    ("phone",         re.compile(r"(Τηλέφωνο\s*:\s*\n)([^\n\r]+)")),
]


In [ ]:
#Signatures: μπορει να εχουμε block titles, ή τετοια μορφη
# Example:
    # Ο Διευθυντής
    # Νίκος Γιαπατζής
    # Ο Επιμελητής
    # Βιργινία Κουρλού κτλ

SIGNATURE_TITLES_BLOCK = re.compile(
    r"("                                     # group(1) = titles block
    r"Ο\s+Διευθυντής\s*[\.\,\;\:\-\–\—]*\s*\n"
    r"(?:(?:"
    r"Ο\s+Επιμελητής"
    r"|Η\s+Εξειδικευόμενη"
    r"|Ο\s+Εξειδικευόμενος"
    r"|Ο\s+(?:Ι|ι)ατρός\s+ΜΕΘ"
    r"|Η\s+(?:Ι|ι)ατρός\s+ΜΕΘ"
    r"|Ο\s+(?:Ι|ι)ατρός"
    r"|Η\s+(?:Ι|ι)ατρός"
    r")\s*[\.\,\;\:\-\–\—]*\s*\n)+"
    r")",
    flags=re.MULTILINE
)

#One-line title matcher (for interleaved signatures)
SIGNATURE_TITLE_LINE = re.compile(
    r"^\s*(?:"
    r"Ο\s+Διευθυντής"
    r"|Ο\s+Επιμελητής"
    r"|Η\s+Εξειδικευόμενη"
    r"|Ο\s+Εξειδικευόμενος"
    r"|[ΟΗ]\s+(?:Ι|ι)ατρός(?:\s+ΜΕΘ)?"
    r")\s*[\.\,\;\:\-\–\—]*\s*$"
)


In [ ]:
#επειδη δεν εχουμε free text δεν χρειαζονται ολα
LABEL_HINTS = [
    "Αρ.", "ΑΜΚΑ", "Διευθυντής", "Καθηγητής",
    "Επώνυμο", "Όνομα", "Διεύθυνση", "Τ.Κ", "Πόλη",
    "Τηλέφωνο", "Ημ/νία", "Ηλικία", "Τμήμα", "Κλινική",
    "Ο Διευθυντής", "Ο Επιμελητής", "Η Εξειδικευόμενη", "Ο Εξειδικευόμενος",
    "Ο Ιατρός ΜΕΘ", "Η Ιατρός ΜΕΘ", "Ο Ιατρός", "Η Ιατρός", "Η ιατρός",
    "ΠΡΟΣΟΧΗ"
]

#για γραμμες με συμβολα
_ONLY_PUNCT = set(string.punctuation) | {"·", "…", "«", "»", "–", "—", "―", "’", "“", "”", "„", "•", "▪", "►", "‐"}

#
def looks_like_label(s: str) -> bool:
    s = (s or "").strip()
    if not s:
        return True
    if s.endswith(":"):
        return True
    s_low = s.lower()
    for h in LABEL_HINTS:
        if h.lower() in s_low and len(s) <= 60:
            return True
    return False

#κοβω τα κενα πριν και μετα
def trim_span(text: str, s: int, e: int):
    chunk = text[s:e]
    left = len(chunk) - len(chunk.lstrip())
    right = len(chunk.rstrip())
    ns = s + left
    ne = s + right
    if ns >= ne:
        return None
    return ns, ne

#αφαιρω σημεια στιξης πριν και μετα
def _strip_edge_punct(value: str) -> str:
    v = (value or "").strip()
    while v and v[0] in _ONLY_PUNCT:
        v = v[1:].lstrip()
    while v and v[-1] in _ONLY_PUNCT:
        v = v[:-1].rstrip()
    return v

def _is_only_punct_line(s: str) -> bool:
    raw = (s or "").strip()
    if not raw:
        return False
    return all(ch in _ONLY_PUNCT for ch in raw)


In [ ]:
PHONE_PAT = re.compile(r"(?:\+30\s*)?(?:69\d|2\d{2})[\s\-]?\d{3}[\s\-]?\d{4}")
#AT_PHONE_TOKEN_PAT = re.compile(r"@[0-9]{5,}")  # [@8854854 etc]]

PHONE_LINE_PAT = re.compile(r"^\s*(?:@?\+30\s*)?(?:69\d|2\d{2})[\d\s\-\(\)]{7,}\s*$")

def is_phone_line(s: str) -> bool:
    return bool(PHONE_LINE_PAT.match(s or ""))

def add_phone_spans_from_value(text, base_s, base_e, preds):
    val = text[base_s:base_e]

    found = []
    found += [(m.start(), m.end()) for m in PHONE_PAT.finditer(val)]
    #found += [(m.start(), m.end()) for m in AT_PHONE_TOKEN_PAT.finditer(val)]

    if not found:
        preds.append((base_s, base_e, "phone"))
        return

    found.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    kept = []
    last_end = -1
    for s, e in found:
        if s >= last_end:
            kept.append((s, e))
            last_end = e

    for s, e in kept:
        preds.append((base_s + s, base_s + e, "phone"))


In [ ]:
def predict_spans_template_regex(template_text: str):
    #Αν το template είναι None ή NaN → το κάνει κενό string
    text = "" if template_text is None or (isinstance(template_text, float) and pd.isna(template_text)) else str(template_text)
    preds = []

    #"Τηλέφωνο:" may have 1-2 lines AFTER the label ----
    for m in re.finditer(r"(Τηλέφωνο\s*:\s*)\n", text):
        pos = m.end()
        offset = pos
        got = 0

        for line in text[pos:].splitlines(True):
            raw = line.strip()
            if not raw:
                offset += len(line)
                continue

            #if raw.startswith("ΠΡΟΣΟΧΗ"):
             #   break

            if is_phone_line(raw): #or AT_PHONE_TOKEN_PAT.search(raw):
                s_line = offset + line.find(raw)
                e_line = s_line + len(raw)
                add_phone_spans_from_value(text, s_line, e_line, preds)
                got += 1
                offset += len(line)
                if got == 2:
                    break
                continue

            break

    # A) fields: NEXT_LINE first, then SAME_LINE
    for lab, pat in (FIELD_NEXT_LINE + FIELD_SAME_LINE):
        for m in pat.finditer(text):
            s, e = m.span(2)
            span = trim_span(text, s, e)
            if not span:
                continue
            s2, e2 = span
            val = text[s2:e2]
            if looks_like_label(val):
                continue

            if lab == "phone":
                add_phone_spans_from_value(text, s2, e2, preds)
            else:
                preds.append((s2, e2, lab))

    # B) signatures: titles-block, then scan names line-by-line until ΠΡΟΣΟΧΗ or end
    for m in SIGNATURE_TITLES_BLOCK.finditer(text):
        pos = m.end(1)
        rest = text[pos:]
        offset = pos

        for line in rest.splitlines(True):
            raw = line.strip()
            if not raw:
                offset += len(line)
                continue

            if raw.startswith("ΠΡΟΣΟΧΗ"):
                break

            if _is_only_punct_line(raw):
                offset += len(line)
                continue

            cleaned = _strip_edge_punct(raw)
            if not cleaned:
                offset += len(line)
                continue
            if looks_like_label(cleaned):
                offset += len(line)
                continue

            idx = raw.find(cleaned)
            if idx == -1:
                s_line = offset + line.find(raw)
                e_line = s_line + len(raw)
            else:
                s_line = offset + line.find(raw) + idx
                e_line = s_line + len(cleaned)

            preds.append((s_line, e_line, "staff_name"))
            offset += len(line)

    # B2) signatures fallback: interleaved TITLE -> NAME
    lines = text.splitlines(True)  # keep \n
    offsets = []
    cur = 0
    for ln in lines:
        offsets.append(cur)
        cur += len(ln)

    i = 0
    while i < len(lines):
        raw = lines[i].strip()

        if raw.startswith("ΠΡΟΣΟΧΗ"):
            break

        if raw and SIGNATURE_TITLE_LINE.match(raw):
            j = i + 1
            # find next meaningful line as name
            while j < len(lines):
                nxt_raw = lines[j].strip()

                if nxt_raw.startswith("ΠΡΟΣΟΧΗ"):
                    j = None
                    break

                if not nxt_raw or _is_only_punct_line(nxt_raw):
                    j += 1
                    continue

                cleaned = _strip_edge_punct(nxt_raw)
                if not cleaned or looks_like_label(cleaned):
                    j = None
                    break

                base_off = offsets[j]
                start_in_line = lines[j].find(nxt_raw)
                if start_in_line < 0:
                    start_in_line = 0

                idx = nxt_raw.find(cleaned)
                if idx < 0:
                    s_line = base_off + start_in_line
                    e_line = s_line + len(nxt_raw)
                else:
                    s_line = base_off + start_in_line + idx
                    e_line = s_line + len(cleaned)

                preds.append((s_line, e_line, "staff_name"))
                break

            if j is not None:
                i = j + 1
                continue

        i += 1

    # C) de-overlap (keep non-overlapping, prefer longer)
    preds.sort(key=lambda x: (x[0], -(x[1]-x[0])))
    kept = []
    last_end = -1
    for s, e, lab in preds:
        if s >= last_end:
            kept.append((s, e, lab))
            last_end = e
    kept.sort(key=lambda x: x[0])
    return kept


In [ ]:
def add_template_pred_spans(df, template_col="template_text", out_col="pred_spans_template"):
    df = df.copy()
    df[out_col] = df[template_col].apply(predict_spans_template_regex)
    return df


In [ ]:
df = add_template_pred_spans(
    df,
    template_col="template_text",
    out_col="pred_spans_template"
)


In [ ]:
def normalize_spans(spans):
    """
    Παίρνει λίστα από (start, end, label)
    και την κάνει set από tuples για exact σύγκριση.
    """
    if spans is None:
        return set()

    out = set()
    for s in spans:
        if len(s) != 3:
            continue
        start, end, label = s
        out.add((int(start), int(end), str(label)))
    return out

def spans_by_label(spans):
    d = {}
    for s, e, lab in spans:
        d.setdefault(lab, set()).add((s, e))
    return d

In [ ]:
def print_classification_report_one_doc(gold_spans, pred_spans, doc_id=None):
    gold = spans_by_label(normalize_spans(gold_spans))
    pred = spans_by_label(normalize_spans(pred_spans))

    labels = sorted(set(gold.keys()) | set(pred.keys()))

    header = f"\n===== CLASSIFICATION REPORT ====="
    if doc_id is not None:
        header += f"\nDocument: {doc_id}"
    print(header)
    print("-" * 50)
    print(f"{'Label':20s} {'TP':>3s} {'FP':>3s} {'FN':>3s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s}")
    print("-" * 50)

    for lab in labels:
        g = gold.get(lab, set())
        p = pred.get(lab, set())

        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 1.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"{lab:20s} {tp:3d} {fp:3d} {fn:3d} {precision:6.2f} {recall:6.2f} {f1:6.2f}")

    print("-" * 50)


In [ ]:
for idx, row in df.iterrows():
    print_classification_report_one_doc(
        gold_spans=row["gold_spans_template"],
        pred_spans=row["pred_spans_template"],
        doc_id=idx
    )



===== CLASSIFICATION REPORT =====
Document: 0
--------------------------------------------------
Label                 TP  FP  FN   Prec    Rec     F1
--------------------------------------------------
first_name             1   0   0   1.00   1.00   1.00
last_name              1   0   0   1.00   1.00   1.00
patient_id             1   0   0   1.00   1.00   1.00
staff_name             2   0   0   1.00   1.00   1.00
--------------------------------------------------

===== CLASSIFICATION REPORT =====
Document: 1
--------------------------------------------------
Label                 TP  FP  FN   Prec    Rec     F1
--------------------------------------------------
first_name             1   0   0   1.00   1.00   1.00
last_name              1   0   0   1.00   1.00   1.00
patient_id             1   0   0   1.00   1.00   1.00
staff_name             2   0   0   1.00   1.00   1.00
--------------------------------------------------

===== CLASSIFICATION REPORT =====
Document: 2
-------------

In [ ]:
def print_report_df(df, gold_col, pred_col):
    totals = {}  # lab -> [tp, fp, fn]

    for _, row in df.iterrows():
        gold = spans_by_label(normalize_spans(row[gold_col]))
        pred = spans_by_label(normalize_spans(row[pred_col]))

        for lab in set(gold) | set(pred):
            g = gold.get(lab, set())
            p = pred.get(lab, set())

            tp = len(g & p)
            fp = len(p - g)
            fn = len(g - p)

            if lab not in totals:
                totals[lab] = [0, 0, 0]
            totals[lab][0] += tp
            totals[lab][1] += fp
            totals[lab][2] += fn

    print("\n===== REPORT (ALL DOCS) =====")
    print(f"{'Label':20s} {'TP':>4s} {'FP':>4s} {'FN':>4s} {'P':>6s} {'R':>6s} {'F1':>6s}")
    print("-" * 50)

    for lab, (tp, fp, fn) in sorted(totals.items()):
        p = tp / (tp + fp) if tp + fp else 1.0
        r = tp / (tp + fn) if tp + fn else 1.0
        f = 2*p*r / (p+r) if p+r else 0.0
        print(f"{lab:20s} {tp:4d} {fp:4d} {fn:4d} {p:6.2f} {r:6.2f} {f:6.2f}")



In [ ]:
print_report_df(df, "gold_spans_template", "pred_spans_template")



===== REPORT (ALL DOCS) =====
Label                  TP   FP   FN      P      R     F1
--------------------------------------------------
address                 2    0    0   1.00   1.00   1.00
first_name             25    0    0   1.00   1.00   1.00
last_name              25    0    0   1.00   1.00   1.00
patient_id             24    0    0   1.00   1.00   1.00
phone                   4    0    0   1.00   1.00   1.00
postal_city             2    0    0   1.00   1.00   1.00
staff_name             66    0    0   1.00   1.00   1.00


In [ ]:
i = 19
row = df.iloc[i]

print("GOLD:")
for s,e,l in row["gold_spans_template"]:
    print(l, "=>", repr(row["template_text"][s:e]))

print("\nPRED:")
for s,e,l in row["pred_spans_template"]:
    print(l, "=>", repr(row["template_text"][s:e]))


In [ ]:
#print(df['template_text'][24])

FREE TEXT

In [ ]:
#load models
nlp_spacy = spacy.load("el_core_news_sm")

ner_xlm = pipeline(
    "token-classification",
    model="Davlan/xlm-roberta-base-ner-hrl",
    aggregation_strategy="simple",
)

ner_gr = pipeline(
    "token-classification",
    model="amichailidis/bert-base-greek-uncased-v1-finetuned-ner",
    aggregation_strategy="simple",
)

NER_MODELS = {
    "spacy_el": "spacy",     # special case (uses nlp_spacy)
    "xlm": ner_xlm,
    "bert_gr": ner_gr,
}


Device set to use cuda:0
Device set to use cuda:0


In [ ]:
#label normalization
def norm_label(lbl: str) -> str:
    if not lbl:
        return "misc"
    x = str(lbl).upper()
    if "PER" in x or "PERSON" in x:
        return "person_name"
    if "ORG" in x:
        return "hospital_name"
    if "LOC" in x or "GPE" in x:
        return "location"
    return "misc"


In [ ]:
#noise filtering
_NOISE_EQ = {"ΜΕΘ", "ΤΕΠ", "ΠΓΝΙ"}

def _norm_noise_token(s: str) -> str:
    s = (s or "").strip().upper()
    s = re.sub(r"[\s\.\-–—\(\)\[\]\{\}]+", "", s)
    return s

def is_noise_entity_text(ent_text: str) -> bool:
    t = _norm_noise_token(ent_text)
    if not t:
        return True
    if t in _NOISE_EQ:
        return True
    if t.startswith("ΠΓΝΙ"):
        return True
    return False


In [ ]:
#phone and patiend id regex
PHONE_REGEX = re.compile(
    r"""
    (?:
        @\d{5,}
        |
        (?:(?:\(\s*\+30\s*\)|\+30)\s*[-–—]?\s*)?
        (?:
            69\d(?:[\s\-]?\d){7}
            |
            2\d{2}(?:[\s\-]?\d){7}
            |
            2\d{3}(?:[\s\-]?\d){6}
        )
    )
    """,
    re.VERBOSE
)

def phone_spans_free_text(text: str):
    if not text:
        return []
    return [(m.start(), m.end(), "phone") for m in PHONE_REGEX.finditer(text)]

PATIENT_ID_REGEX = re.compile(r"(?<!\d)\d{8}(?!\d)")

def patient_id_spans_free_text(text: str):
    if not text:
        return []
    return [(m.start(), m.end(), "patient_id") for m in PATIENT_ID_REGEX.finditer(text)]


In [ ]:
#αν καποιο σπαν πεσει σε τηλ το αγνοουμε
def _overlaps_any(s, e, ranges):
    for a, b in ranges:
        if max(s, a) < min(e, b):
            return True
    return False


In [ ]:
#Κόβει μεγάλο text σε chunks, τρέχει NER σε κάθε chunk, και φτιάχνει spans πίσω στο original κείμενο. Κυριως για grbert
def ner_hf_spans_chunked(text: str, ner_pipe, chunk_chars=1200, overlap=200):
    if not text:
        return []
    spans = []
    n = len(text)
    i = 0

    while i < n:
        j = min(n, i + chunk_chars)
        chunk = text[i:j]
        ents = ner_pipe(chunk)

        for ent in ents:
            s = i + int(ent["start"])
            e = i + int(ent["end"])
            lab = norm_label(ent.get("entity_group") or ent.get("entity"))
            spans.append((s, e, lab))

        if j == n:
            break
        i = j - overlap

    # keep non-overlapping, prefer longer
    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    kept = []
    last_end = -1
    for s, e, lab in spans:
        if s >= last_end:
            kept.append((s, e, lab))
            last_end = e
    kept.sort(key=lambda x: x[0])
    return kept


In [ ]:
#ner extractors
def free_text_spans_spacy(text: str):
    text = "" if text is None or (isinstance(text, float) and pd.isna(text)) else str(text)

    spans = []
    spans += phone_spans_free_text(text)
    spans += patient_id_spans_free_text(text)

    protected = [(s, e) for s, e, _ in spans]

    doc = nlp_spacy(text)
    for ent in doc.ents:
        s, e = ent.start_char, ent.end_char
        if _overlaps_any(s, e, protected):
            continue
        if is_noise_entity_text(ent.text):
            continue
        spans.append((s, e, norm_label(ent.label_)))

    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    return spans


def free_text_spans_hf(text: str, ner_pipe):
    text = "" if text is None or (isinstance(text, float) and pd.isna(text)) else str(text)

    spans = []
    spans += phone_spans_free_text(text)
    spans += patient_id_spans_free_text(text)

    protected = [(s, e) for s, e, _ in spans]

    ents = ner_hf_spans_chunked(text, ner_pipe)
    for s, e, lab in ents:
        if _overlaps_any(s, e, protected):
            continue
        if is_noise_entity_text(text[s:e]):
            continue
        spans.append((s, e, lab))

    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    return spans


In [ ]:
def predict_spans_free_text(text: str, model_key: str):
    model = NER_MODELS[model_key]

    if model == "spacy":
        return free_text_spans_spacy(text)
    else:
        return free_text_spans_hf(text, model)


In [ ]:
FREE_TEXT_COL = "free_text"

for model_key in NER_MODELS.keys():
    out_col = f"free_pred_spans__{model_key}"
    df[out_col] = df[FREE_TEXT_COL].apply(lambda t: predict_spans_free_text(t, model_key))

df[[FREE_TEXT_COL] + [c for c in df.columns if c.startswith("free_pred_spans__")]].head(2)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


,free_text,free_pred_spans__spacy_el,free_pred_spans__xlm,free_pred_spans__bert_gr
0,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ – ΠΟΡΕΙΑ ΝΟΣΟΥ\nΑτομι...,"[(298, 307, hospital_name), (366, 375, hospita...","[(297, 307, hospital_name), (309, 320, locatio...","[(243, 245, hospital_name), (298, 321, hospita..."
1,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\n\nΠρογραμματισμένη ε...,"[(60, 75, person_name), (1727, 1735, hospital_...","[(55, 75, hospital_name), (77, 89, location), ...","[(56, 58, hospital_name), (58, 89, hospital_na..."


In [ ]:
#model_key='xlm'

In [ ]:
df.head()

RESULTS

In [ ]:
l

In [ ]:
# Example:
print('XLM-R')
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_spans__xlm", tau=0.8))
print('spacy')
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_spans__spacy_el", tau=0.8))
print('grbert')
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_spans__bert_gr", tau=0.8))

XLM-R
{'tp': 102, 'fp': 104, 'fn': 1, 'tp_gold': 49, 'precision': 0.49514563106796117, 'recall': 0.98, 'f1': 0.6578912728708701, 'tau': 0.8}
spacy
{'tp': 48, 'fp': 353, 'fn': 24, 'tp_gold': 26, 'precision': 0.11970074812967581, 'recall': 0.52, 'f1': 0.19460470918446904, 'tau': 0.8}
grbert
{'tp': 41, 'fp': 271, 'fn': 11, 'tp_gold': 39, 'precision': 0.13141025641025642, 'recall': 0.78, 'f1': 0.22492614995076665, 'tau': 0.8}


In [ ]:
print('XLM-R')
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_spans__xlm", tau=0.9))
print('spacy')
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_spans__spacy_el", tau=0.9))
print('grbert')
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_spans__bert_gr", tau=0.9))

XLM-R
{'tp': 102, 'fp': 104, 'fn': 11, 'tp_gold': 39, 'precision': 0.49514563106796117, 'recall': 0.78, 'f1': 0.6057560529922339, 'tau': 0.9}
spacy
{'tp': 48, 'fp': 353, 'fn': 27, 'tp_gold': 23, 'precision': 0.11970074812967581, 'recall': 0.46, 'f1': 0.18996816656629095, 'tau': 0.9}
grbert
{'tp': 41, 'fp': 271, 'fn': 12, 'tp_gold': 38, 'precision': 0.13141025641025642, 'recall': 0.76, 'f1': 0.2240759384438372, 'tau': 0.9}


In [ ]:
print('XLM-R')
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_spans__xlm", tau=1))
print('spacy')
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_spans__spacy_el", tau=1))
print('grbert')
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_spans__bert_gr", tau=1))

XLM-R
{'tp': 102, 'fp': 104, 'fn': 24, 'tp_gold': 26, 'precision': 0.49514563106796117, 'recall': 0.52, 'f1': 0.5072685539403213, 'tau': 1}
spacy
{'tp': 48, 'fp': 353, 'fn': 27, 'tp_gold': 23, 'precision': 0.11970074812967581, 'recall': 0.46, 'f1': 0.18996816656629095, 'tau': 1}
grbert
{'tp': 41, 'fp': 271, 'fn': 24, 'tp_gold': 26, 'precision': 0.13141025641025642, 'recall': 0.52, 'f1': 0.20980122023223777, 'tau': 1}


In [ ]:
def print_free_text_FNs_with_preds(
    df,
    gold_col,
    pred_col,
    text_col,
    doc_col="file",
    tau=0.8,
):
    total_fn = 0

    for _, r in df.iterrows():
        gold = r[gold_col] or []
        pred = r[pred_col] or []
        text = r[text_col] or ""
        doc  = r.get(doc_col, "<doc>")

        for gs, ge, glab in gold:
            glen = ge - gs
            if glen <= 0:
                continue

            # find overlapping predicted spans
            overlaps = []
            covered = 0
            for ps, pe, plab in pred:
                s = max(gs, ps)
                e = min(ge, pe)
                if s < e:
                    overlaps.append((ps, pe, plab, text[ps:pe]))
                    covered += (e - s)

            cov = covered / glen

            if cov < tau:
                total_fn += 1
                print("\nFILE:", doc)
                print(f"FN GOLD | label={glab} | coverage={cov:.2f}")
                print("  GOLD TEXT:", repr(text[gs:ge]))

                if overlaps:
                    print("  OVERLAPPING PREDICTED SPANS:")
                    for ps, pe, plab, ptxt in overlaps:
                        print(f"    - {plab:<15} | {repr(ptxt)}")
                else:
                    print("  NO PREDICTED SPANS OVERLAPPED")

    print("\nTOTAL FN:", total_fn)


In [ ]:
print_free_text_FNs_with_preds(
    df,
    gold_col="gold_spans_free_text",
    #pred_col="free_pred_spans__xlm",
    #pred_col="free_pred_spans__spacy_el",
    pred_col="free_pred_spans__bert_gr",
    text_col="free_text",
    doc_col="file",
    tau=1
)

In [ ]:
from IPython.display import display, HTML

PASTEL_COLORS = {
  "person_name":   "#c084fc",
    "staff_name":    "#60a5fa",
    "hospital_name": "#4ade80",
    "location":      "#60a5fa",
    "phone":         "#f87171",
    "address":       "#f7f7f7",
    "postal_city":   "#93c5fd",
    "patient_id":    "#f6f6f6",
    "misc":          "#93c5fd",
}


def render_spans_html(text, spans, colors=PASTEL_COLORS):
    spans = sorted(spans, key=lambda x: x[0])
    html = ""
    i = 0

    for s, e, lab in spans:
        html += text[i:s]
        color = colors.get(lab, "#f0f0f0")
        html += (
            f'<span style="background:{color}; '
            f'padding:2px 4px; '
            f'border-radius:4px; '
            f'white-space:pre-wrap;">'
            f'{text[s:e]}'
            f'<sup style="font-size:0.7em; margin-left:4px; color:#666;">{lab}</sup>'
            f'</span>'
        )
        i = e

    html += text[i:]
    display(HTML(html.replace("\n", "<br>")))



In [ ]:
i = 0
row = df.iloc[i]

print("GOLD:")
for s,e,l in row["gold_spans_free_text"]:
    print(l, "=>", repr(row["free_text"][s:e]))

print("\nPRED:")
for s,e,l in row["free_pred_spans__xlm"]:
#for s,e,l in row["free_pred_spans__spacy_el"]:
#for s,e,l in row['free_pred_spans__bert_gr']:
    print(l, "=>", repr(row["free_text"][s:e]))


GOLD:
hospital_name => 'ΓΝ Αθηνών «Ιπποκράτειο»'
hospital_name => 'ΓΝ Αθηνών «Ιπποκράτειο»'
hospital_name => 'ΓΝ Αθηνών «Ιπποκράτειο»'

PRED:
hospital_name => ' ΓΝ Αθηνών'
location => 'Ιπποκράτειο'
hospital_name => ' Γ'
location => 'Ν Αθηνών'
location => 'Ιπποκράτειο'
location => ' ΓΝ Αθηνών'
location => 'Ιπποκράτειο'
location => ' Π'
hospital_name => 'ΓΝΙ'
location => ' Μ'


In [ ]:
render_spans_html(
    row["free_text"],
    row["free_pred_spans__xlm"]
    #row["free_pred_spans__spacy_el"]
    #row['free_pred_spans__bert_gr']



)


LLAMA

In [ ]:
%%capture
!pip install boto3
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

In [ ]:
from google.colab import files
import json
print ('Upload the `aws-credentials.json` file: ')
files.upload()
credentials = json.load(open('aws-credentials.json'))

Upload the `aws-credentials.json` file: 


Saving aws-credentials.json to aws-credentials (1).json


In [ ]:
%%capture
# Initialize the boto3 client for Bedrock
bedrock_client = boto3.client(
    'bedrock',
    aws_access_key_id=credentials['aws_access_key_id'],
    aws_secret_access_key=credentials['aws_secret_access_key'],
    region_name=credentials['aws_region']
)
bedrock_client.list_foundation_models()['modelSummaries']

In [ ]:
# Use the native inference API to send a text message to Meta Llama 3.
# Create a Bedrock Runtime client in the AWS Region of your choice.
client = boto3.client("bedrock-runtime",
                      aws_access_key_id=credentials['aws_access_key_id'],
                      aws_secret_access_key=credentials['aws_secret_access_key'],
                      region_name=credentials['aws_region'])

model_id = "meta.llama3-70b-instruct-v1:0"

In [ ]:
def mask_ranges_keep_length(text: str, ranges, mask_char="█"):
    if not text:
        return ""
    t = list(text)
    for s, e in ranges:
        s = max(0, min(len(t), int(s)))
        e = max(0, min(len(t), int(e)))
        for i in range(s, e):
            t[i] = mask_char
    return "".join(t)


In [ ]:
LLAMA_LIST_INSTRUCTION = """Είσαι εργαλείο NER για ανωνυμοποίηση ιατρικού free text στα Ελληνικά.
Βρες ευαίσθητες οντότητες που πρέπει να καλυφθούν (PII/PHI) ΕΚΤΟΣ από τηλέφωνα και αριθμητικά IDs.

Επιστροφή ΜΟΝΟ σε έγκυρο JSON, χωρίς εξηγήσεις/markdown.

Κανόνες:
- Μην επιστρέφεις τίποτα που περιέχει '*' ή '█' (είναι ήδη masked).
- Μην επιστρέφεις ΜΕΘ, ΤΕΠ, ΠΓΝΙ (ούτε παραλλαγές).
- Μην επιστρέφεις spans < 3 χαρακτήρες.
- Επιστροφή με ακριβή substrings όπως εμφανίζονται στο κείμενο.

Schema:
{"entities":[{"label":"person_name|hospital_name|location|misc","text":"..."}]}

Αν δεν βρεις τίποτα:
{"entities":[]}
"""

def build_llama_list_prompt(masked_text: str):
    return f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{LLAMA_LIST_INSTRUCTION}

ΚΕΙΜΕΝΟ:
{masked_text}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""





In [ ]:
def llama_call_list(masked_text: str, client, model_id: str, max_len=900):
    prompt = build_llama_list_prompt(masked_text)
    body = json.dumps({"prompt": prompt, "max_gen_len": max_len, "temperature": 0.0})
    resp = client.invoke_model(modelId=model_id, body=body)
    out = json.loads(resp["body"].read())
    return out.get("generation", "")


In [ ]:
_NOISE_EQ = {"ΜΕΘ", "ΤΕΠ", "ΠΓΝΙ"}

def _norm_noise_token(s: str) -> str:
    s = (s or "").strip().upper()
    s = re.sub(r"[\s\.\-–—\(\)\[\]\{\}]+", "", s)
    return s

def is_noise_entity_text(ent_text: str) -> bool:
    t = _norm_noise_token(ent_text)
    if not t:
        return True
    if t in _NOISE_EQ:
        return True
    if t.startswith("ΠΓΝΙ"):
        return True
    return False

TAG_OPEN = {
    "[[PERSON]]": "person_name",
    "[[HOSP]]": "hospital_name",
    "[[LOC]]": "location",
    "[[MISC]]": "misc",
}
TAG_CLOSE = {
    "person_name": "[[/PERSON]]",
    "hospital_name": "[[/HOSP]]",
    "location": "[[/LOC]]",
    "misc": "[[/MISC]]",
}

def parse_tagged_text_to_spans(tagged_text: str):
    """
    Επιστρέφει:
      clean_text: το κείμενο χωρίς tags
      spans: [(start,end,label), ...] offsets πάνω στο clean_text
    """
    if tagged_text is None:
        return "", []

    t = str(tagged_text)
    clean = []
    spans = []
    i = 0
    out_pos = 0

    while i < len(t):
        # check for any opening tag at position i
        opened = None
        for tag, lab in TAG_OPEN.items():
            if t.startswith(tag, i):
                opened = (tag, lab)
                break

        if opened:
            tag, lab = opened
            i += len(tag)
            start = out_pos

            close_tag = TAG_CLOSE[lab]
            j = t.find(close_tag, i)
            if j == -1:
                # no close tag -> stop parsing safely
                break

            content = t[i:j]
            clean.append(content)
            out_pos += len(content)

            end = out_pos
            spans.append((start, end, lab))

            i = j + len(close_tag)
            continue

        # normal char
        clean.append(t[i])
        i += 1
        out_pos += 1

    return "".join(clean), spans


In [ ]:
def parse_llama_entity_list(raw: str):
    raw = (raw or "").strip()
    i = raw.find("{")
    j = raw.rfind("}")
    if i == -1 or j == -1 or j <= i:
        return []

    try:
        obj = json.loads(raw[i:j+1])
    except Exception:
        return []

    ents = obj.get("entities", [])
    cleaned = []
    for ent in ents:
        try:
            lab = str(ent.get("label", "misc"))
            txt = str(ent.get("text", "")).strip()
        except Exception:
            continue

        if not txt or len(txt) < 3:
            continue
        if "*" in txt or "█" in txt:
            continue
        if is_noise_entity_text(txt):
            continue

        cleaned.append((txt, lab))
    return cleaned


In [ ]:
def find_all_spans_exact(text: str, needle: str):
    if not needle:
        return []
    return [(m.start(), m.end()) for m in re.finditer(re.escape(needle), text)]

def llama_entities_to_spans(original_text: str, entity_list):
    spans = []
    for needle, lab in entity_list:
        for s, e in find_all_spans_exact(original_text, needle):
            spans.append((s, e, lab))

    # de-overlap (prefer longer)
    spans.sort(key=lambda x: (x[0], -(x[1]-x[0])))
    kept = []
    last_end = -1
    for s, e, lab in spans:
        if s >= last_end:
            kept.append((s, e, lab))
            last_end = e
    kept.sort(key=lambda x: x[0])
    return kept



In [ ]:
def free_text_spans_llama(text: str, client, model_id: str):
    text = "" if text is None else str(text)

    # 1) rules first
    spans = []
    spans += phone_spans_free_text(text)
    spans += patient_id_spans_free_text(text)

    protected = [(s,e) for s,e,_ in spans]
    masked = mask_ranges_keep_length(text, protected, mask_char="*")

    # 2) llama list entities
    raw = llama_call_list(masked, client=client, model_id=model_id)
    entity_list = parse_llama_entity_list(raw)

    # 3) convert entity texts -> spans on ORIGINAL text
    llama_spans = llama_entities_to_spans(text, entity_list)

    spans += llama_spans
    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    return spans, raw, entity_list


In [ ]:
i = 0
text = str(df.loc[i, "free_text"])

spans, raw, entity_list = free_text_spans_llama(text, client=client, model_id=model_id)

print("Raw (first 800):\n", raw[:800])
print("\nEntity list (first 20):", entity_list[:20])
print("\nSpans (first 10):", spans[:10])

for s,e,lab in spans[:10]:
    print(lab, repr(text[s:e]))



Raw (first 800):
 

{"entities":[{"label":"person_name","text":"Εliquis"},{"label":"hospital_name","text":"ΓΝ Αθηνών «Ιπποκράτειο»"},{"label":"location","text":"ΓΝ Αθηνών «Ιπποκράτειο»"},{"label":"location","text":"ΠΓΝΙ"}]}

Entity list (first 20): [('Εliquis', 'person_name'), ('ΓΝ Αθηνών «Ιπποκράτειο»', 'hospital_name'), ('ΓΝ Αθηνών «Ιπποκράτειο»', 'location')]

Spans (first 10): [(298, 321, 'hospital_name'), (366, 389, 'hospital_name'), (634, 657, 'hospital_name')]
hospital_name 'ΓΝ Αθηνών «Ιπποκράτειο»'
hospital_name 'ΓΝ Αθηνών «Ιπποκράτειο»'
hospital_name 'ΓΝ Αθηνών «Ιπποκράτειο»'


In [ ]:
df["free_pred_llama"] = df['free_text'].apply(lambda t: free_text_spans_llama(t, client=client, model_id=model_id)[0])

print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_llama", tau=0.8))


{'tp': 52, 'fp': 61, 'fn': 9, 'tp_gold': 41, 'precision': 0.46017699115044247, 'recall': 0.82, 'f1': 0.5895202543896032, 'tau': 0.8}


In [ ]:
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_llama", tau=0.9))


{'tp': 52, 'fp': 61, 'fn': 11, 'tp_gold': 39, 'precision': 0.46017699115044247, 'recall': 0.78, 'f1': 0.5788497217068646, 'tau': 0.9}


In [ ]:
print(global_label_agnostic_coverage(df, "gold_spans_free_text", "free_pred_llama", tau=1))

{'tp': 52, 'fp': 60, 'fn': 16, 'tp_gold': 34, 'precision': 0.4642857142857143, 'recall': 0.68, 'f1': 0.5518102372034956, 'tau': 1}


In [ ]:
i = 13
row = df.iloc[i]

print("GOLD:")
for s,e,l in row["gold_spans_free_text"]:
    print(l, "=>", repr(row["free_text"][s:e]))

print("\nPRED:")
for s,e,l in row["free_pred_llama"]:
#for s,e,l in row["free_pred_spans__xlm"]:
#for s,e,l in row["free_pred_spacy"]:
#for s,e,l in row["free_pred_grbert"]:
    print(l, "=>", repr(row["free_text"][s:e]))


GOLD:
hospital_name => 'ΓΝ Λαμίας'
hospital_name => 'ΓΝΛ'
hospital_name => 'ΓΝΛ'
hospital_name => 'ΓΝ Λαμίας'
hospital_name => 'ΓΝΛ'

PRED:
person_name => 'Ασθενής'
hospital_name => 'ΓΝ Λαμίας'
hospital_name => 'ΓΝ Λαμίας'


In [ ]:
df["free_text"][23]

'ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\nΚαταπληξία-ισχαιμική νέκρωση εντέρου\nΑτομικό αναμνηστικό: στεφανιαία νόσος, σακχαρώδης διαβήτης ΙΙ, ευμεγέθης διαφραγματοκήλη (χ/ο προ 35 έτη), ρευματική πολυμυαλγία υπό κορτιζόνη.\n\nΔΙΑΓΝΩΣΗ ΕΞΟΔΟΥ\n\nΧειρουργηθείσα ισχαιμική νέκρωση εντέρου- σηπτική καταπληξία –πολυοργανική ανεπάρκεια.\n\n\nΠΟΡΕΙΑ ΝΟΣΟΥ\nΕΠΕΜΒΑΣΕΙΣ\nΗ ασθενής 26/5/2024 διακομίζεται από τους οικείους της στο Γ.Ν. Σερρών αιτιώμενη αιφνίδιας έναρξης κοιλιακό άλγος( κυρίως υπογαστρίου) και συνοδούς εμέτους από ωρών . κατά την κλινική εξέταση διαπιστώθηκε δικτυωτή πελίδνωση κορμού και κάτω άκρων  και υπόταση. Τέθηκε κεντρική φλεβική γραμμή ,χορηγήθηκαν υγρά και αγγειοσυσπαστικά και η ασθενής διακομίστηκε στο ΤΕΠ του νοσοκομείου ΠΓΝΙ.\nΤΕΠ ΠΓΝΙ:\nΚατα την εισαγωγή της στο ΤΕΠ ήταν σε καταπληξία με την ίδια κλινική εικόνα ( εμμένουσα δικτυωτή πελίδνωση –κοιλιακό άλγος διάχυτο –στάγδην χορήγηση ινοτρόπων). Χορηγήθηκε αντιβιοτική αγωγή και εισήχθη στην παθολογική κλινική αφού πρώτα διενεγήθηκε  αξ

In [ ]:
print_free_text_FNs_with_preds(
    df,
    gold_col="gold_spans_free_text",
    pred_col="free_pred_llama",
    text_col="free_text",
    doc_col="file",
    tau=0.8
)


FILE: Ασθ14.docx
FN GOLD | label=hospital_name | coverage=0.00
  GOLD TEXT: 'ΓΝΛ'
  NO PREDICTED SPANS OVERLAPPED

FILE: Ασθ14.docx
FN GOLD | label=hospital_name | coverage=0.00
  GOLD TEXT: 'ΓΝΛ'
  NO PREDICTED SPANS OVERLAPPED

FILE: Ασθ14.docx
FN GOLD | label=hospital_name | coverage=0.00
  GOLD TEXT: 'ΓΝΛ'
  NO PREDICTED SPANS OVERLAPPED

FILE: Ασθ15.docx
FN GOLD | label=hospital_name | coverage=0.52
  GOLD TEXT: 'ΓΝ Αθηνών «Γ. Γεννηματάς»'
  OVERLAPPING PREDICTED SPANS:
    - person_name     | 'Γ. Γεννηματάς'

FILE: Ασθ15.docx
FN GOLD | label=hospital_name | coverage=0.52
  GOLD TEXT: 'ΓΝ Αθηνών «Γ. Γεννηματάς»'
  OVERLAPPING PREDICTED SPANS:
    - person_name     | 'Γ. Γεννηματάς'

FILE: Ασθ15.docx
FN GOLD | label=hospital_name | coverage=0.52
  GOLD TEXT: 'ΓΝ Αθηνών «Γ. Γεννηματάς»'
  OVERLAPPING PREDICTED SPANS:
    - person_name     | 'Γ. Γεννηματάς'

FILE: Ασθ17.docx
FN GOLD | label=hospital_name | coverage=0.00
  GOLD TEXT: 'ΠΓΝ Βόλου «Αχιλλοπούλειο»'
  NO PREDICTED SPANS O

In [ ]:
render_spans_html(
    row["free_text"],
    #row["free_pred_spans__xlm"]
    #row["free_pred_spans__spacy_el"]
    #row['free_pred_spans__bert_gr']
    row["free_pred_llama"]



)

In [ ]:
import pandas as pd

def global_cov1_metrics(df, gold_col, pred_col, beta=2.0):
    TPp = FP = TPg = FN = 0

    for _, r in df.iterrows():
        gold = [(s, e) for s, e, _ in (r.get(gold_col) or [])]
        pred = [(s, e) for s, e, _ in (r.get(pred_col) or [])]

        # Recall@1: gold detected if 100% of gold is covered by UNION(pred spans)
        tp_gold = 0
        for g in gold:
            glen = g[1] - g[0]
            if glen <= 0:
                continue
            covered = _union_covered_len(g, pred)
            if covered == glen:  # tau=1.0
                tp_gold += 1
        fn = len(gold) - tp_gold

        # Precision: predicted span is TP if it overlaps ANY gold span
        tp_pred = 0
        for p in pred:
            if any(_intersection_len(p, g) > 0 for g in gold):
                tp_pred += 1
        fp = len(pred) - tp_pred

        TPp += tp_pred; FP += fp; TPg += tp_gold; FN += fn

    precision = TPp / (TPp + FP) if (TPp + FP) else 0.0
    recall    = TPg / (TPg + FN) if (TPg + FN) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    b2 = beta * beta
    f2 = ((1 + b2) * precision * recall / (b2 * precision + recall)) if (b2 * precision + recall) else 0.0

    return {
        "tp_pred": TPp, "fp": FP,
        "tp_gold": TPg, "fn": FN,
        "precision": precision, "recall": recall, "f1": f1, "f2": f2
    }

def cov1_summary_table(df, gold_col, model_pred_cols, beta=2.0, decimals=3):
    rows = []
    for model, pred_col in model_pred_cols.items():
        g = global_cov1_metrics(df, gold_col, pred_col, beta=beta)
        rows.append({
            "model": model,
            #"tp_pred": g["tp_pred"],
            #"fp": g["fp"],
            #"tp_gold": g["tp_gold"],
            #"fn": g["fn"],
            "precision": round(g["precision"], decimals),
            "recall": round(g["recall"], decimals),
            "f1": round(g["f1"], decimals),
            "f2": round(g["f2"], decimals),
        })
    return pd.DataFrame(rows).sort_values("model")


In [ ]:
MODEL_PREDS = {
    "XLM-R":   "free_pred_spans__xlm",
    "spaCy-el":"free_pred_spans__spacy_el",
    "GR-BERT":"free_pred_spans__bert_gr",
    "Llama3":  "free_pred_llama",
}

cov1_table = cov1_summary_table(df, "gold_spans_free_text", MODEL_PREDS, beta=2.0, decimals=3)
display(cov1_table)


,model,precision,recall,f1,f2
2,GR-BERT,0.131,0.52,0.210,0.327
3,Llama3,0.460,0.68,0.549,0.621
0,XLM-R,0.495,0.52,0.507,0.515
1,spaCy-el,0.120,0.46,0.190,0.293


EXACT SPAN

In [ ]:
import pandas as pd

def label_agnostic_exact_span_scores(gold_spans, pred_spans):
    """
    Label-agnostic exact span matching.
    gold_spans/pred_spans: list of (start, end, label) or None.
    Returns tp, fp, fn, precision, recall, f1, f2
    """
    gold_set = set((s, e) for s, e, _ in (gold_spans or []))
    pred_set = set((s, e) for s, e, _ in (pred_spans or []))

    matched = gold_set & pred_set

    tp = len(matched)
    fp = len(pred_set) - tp
    fn = len(gold_set) - tp

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0

    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    beta = 2.0
    f2 = ((1 + beta**2) * precision * recall / ((beta**2 * precision) + recall)) if ((beta**2 * precision) + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f2": f2,
    }

def global_label_agnostic_exact_span(df, gold_col, pred_col):
    tot_tp = tot_fp = tot_fn = 0

    for _, r in df.iterrows():
        s = label_agnostic_exact_span_scores(r.get(gold_col), r.get(pred_col))
        tot_tp += s["tp"]
        tot_fp += s["fp"]
        tot_fn += s["fn"]

    precision = tot_tp / (tot_tp + tot_fp) if (tot_tp + tot_fp) else 0.0
    recall    = tot_tp / (tot_tp + tot_fn) if (tot_tp + tot_fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    beta = 2.0
    f2 = ((1 + beta**2) * precision * recall / ((beta**2 * precision) + recall)) if ((beta**2 * precision) + recall) else 0.0

    return {
        "tp": tot_tp,
        "fp": tot_fp,
        "fn": tot_fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f2": f2,
    }

def build_exact_span_table_and_export(
    df,
    gold_col,
    model_pred_cols,          # dict: {"XLM-R": "free_pred_spans__xlm", ...}
    out_path="exact_span_label_agnostic.xlsx",
    decimals=3,
    as_percent=False
):
    rows = []
    for model_name, pred_col in model_pred_cols.items():
        g = global_label_agnostic_exact_span(df, gold_col, pred_col)
        rows.append({
            "model": model_name,
            "tp": g["tp"],
            "fp": g["fp"],
            "fn": g["fn"],
            "precision": g["precision"],
            "recall": g["recall"],
            "f1": g["f1"],
            "f2": g["f2"],
        })

    tdf = pd.DataFrame(rows).sort_values("model")

    metric_cols = ["precision", "recall", "f1", "f2"]
    if as_percent:
        tdf[metric_cols] = (tdf[metric_cols] * 100).round(1)
    else:
        tdf[metric_cols] = tdf[metric_cols].round(decimals)

    slides_df = tdf[["model", "precision", "recall", "f1", "f2"]].copy()

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        tdf.to_excel(writer, sheet_name="exact_span", index=False)
        ws = writer.book["exact_span"]
        ws.freeze_panes = "A2"

        slides_df.to_excel(writer, sheet_name="slides", index=False)
        ws2 = writer.book["slides"]
        ws2.freeze_panes = "A2"

    return tdf, slides_df, out_path


In [ ]:
MODEL_PREDS = {
    "XLM-R":   "free_pred_spans__xlm",
    "spaCy-el":"free_pred_spans__spacy_el",
    "GR-BERT":"free_pred_spans__bert_gr",
    "Llama3":  "free_pred_llama",
}

tdf, slides_df, out_path = build_exact_span_table_and_export(
    df=df,
    gold_col="gold_spans_free_text",
    model_pred_cols=MODEL_PREDS,
    out_path="exact_span_label_agnostic.xlsx",
    decimals=3,
    as_percent=False
)

print("Saved:", out_path)
display(tdf)
display(slides_df)


Saved: exact_span_label_agnostic.xlsx


,model,tp,fp,fn,precision,recall,f1,f2
2,GR-BERT,25,287,25,0.080,0.50,0.138,0.244
3,Llama3,33,79,17,0.295,0.66,0.407,0.529
0,XLM-R,5,201,45,0.024,0.10,0.039,0.062
1,spaCy-el,20,381,30,0.050,0.40,0.089,0.166


,model,precision,recall,f1,f2
2,GR-BERT,0.080,0.50,0.138,0.244
3,Llama3,0.295,0.66,0.407,0.529
0,XLM-R,0.024,0.10,0.039,0.062
1,spaCy-el,0.050,0.40,0.089,0.166


Exact span without spaces

In [ ]:
from IPython.display import display, HTML

def _span_text(text, s, e):
    return text[s:e]

def _clip_spans(spans, win_s, win_e):
    out = []
    for s,e,l in (spans or []):
        s2, e2 = max(s, win_s), min(e, win_e)
        if s2 < e2:
            out.append((s2-win_s, e2-win_s, l))
    return out

def _choose_window_around(text, s, e, context=60):
    n = len(text)
    win_s = max(0, s - context)
    win_e = min(n, e + context)
    return win_s, win_e

def classify_gold_span(text, gold_span, pred_spans):
    """
    gold_span: (gs, ge, glabel)
    pred_spans: list of (ps, pe, plabel)
    Returns: dict with coverage, exact_match (bool), any_overlap (bool)
    """
    gs, ge, gl = gold_span
    gold_len = max(0, ge-gs)
    pred = [(s,e) for s,e,_ in (pred_spans or [])]

    covered = _union_covered_len((gs,ge), pred)
    coverage = (covered / gold_len) if gold_len else 0.0

    pred_set = set(pred)
    exact_match = (gs,ge) in pred_set
    any_overlap = any(_intersection_len((gs,ge), p) > 0 for p in pred)

    return {
        "coverage": coverage,
        "exact": exact_match,
        "overlap": any_overlap
    }

def collect_xlmr_examples(df, gold_col, pred_col, text_col="free_text", doc_col=None, max_per_group=3):
    """
    Returns dict with lists of example dicts.
    Each example is one GOLD span (not whole doc) so we can zoom around it.
    """
    groups = {
        "covered_not_exact": [],  # coverage==1 but exact==False
        "exact": [],              # exact==True
        "not_covered": [],        # coverage<1 (FN under coverage@1)
    }

    for idx, row in df.iterrows():
        text = row[text_col]
        gold_spans = row.get(gold_col) or []
        pred_spans = row.get(pred_col) or []
        doc = row.get(doc_col) if doc_col else None

        for g in gold_spans:
            info = classify_gold_span(text, g, pred_spans)

            ex = {
                "row_idx": idx,
                "doc": doc,
                "gold": g,
                "coverage": info["coverage"],
                "exact": info["exact"],
                "overlap": info["overlap"],
            }

            if info["exact"]:
                if len(groups["exact"]) < max_per_group:
                    groups["exact"].append(ex)
            elif info["coverage"] >= 1.0 - 1e-12:
                if len(groups["covered_not_exact"]) < max_per_group:
                    groups["covered_not_exact"].append(ex)
            else:
                if len(groups["not_covered"]) < max_per_group:
                    groups["not_covered"].append(ex)

        # early stop if all full
        if all(len(v) >= max_per_group for v in groups.values()):
            break

    return groups


In [ ]:
def render_gold_vs_preds_snippet(df, ex, gold_col, pred_col, text_col="free_text", context=60):
    row = df.loc[ex["row_idx"]]
    text = row[text_col]
    pred_spans = row.get(pred_col) or []

    gs, ge, gl = ex["gold"]
    win_s, win_e = _choose_window_around(text, gs, ge, context=context)
    snippet = text[win_s:win_e]

    # GOLD span as a single label
    gold_snip_spans = _clip_spans([(gs, ge, "GOLD")], win_s, win_e)

    # PRED spans clipped
    pred_snip_spans = _clip_spans(pred_spans, win_s, win_e)

    # simple renderer
    def esc(x):
        return (x.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;"))

    def render(snippet, spans, color_map, show_label=False):
        spans = sorted(spans, key=lambda x: x[0])
        out, i = "", 0
        for s,e,l in spans:
            out += esc(snippet[i:s])
            col = color_map.get(l, "#22c55e")
            lab = f"<sup style='margin-left:4px; font-size:0.7em; color:#94a3b8;'>{l}</sup>" if show_label else ""
            out += f"<span style='background:{col}; color:#111827; font-weight:900; padding:1px 4px; border-radius:6px;'>{esc(snippet[s:e])}{lab}</span>"
            i = e
        out += esc(snippet[i:])
        return out.replace("\n","<br>")

    gold_html = render(snippet, gold_snip_spans, {"GOLD":"#fbbf24"}, show_label=False)
    pred_html = render(snippet, pred_snip_spans, {}, show_label=False)  # all preds green by default

    header = f"""
    <div style="font-family:system-ui; color:#e5e7eb; font-weight:900; margin-bottom:6px;">
      row={ex["row_idx"]} {(" | doc="+str(ex["doc"])) if ex["doc"] else ""}
      | coverage={ex["coverage"]:.2f} | exact={ex["exact"]} | overlap={ex["overlap"]}
    </div>
    <div style="font-family:system-ui; color:#94a3b8; margin-bottom:6px;">
      GOLD label={gl} | GOLD text: <span style="color:#e5e7eb;">{esc(text[gs:ge])}</span>
    </div>
    """

    html = f"""
    <div style="background:#0b1220; padding:12px; border-radius:16px; margin:10px 0;">
      {header}
      <div style="display:grid; grid-template-columns:1fr 1fr; gap:12px;">
        <div style="background:#0f172a; border:1px solid #1f2937; border-radius:12px; padding:10px;">
          <div style="color:#94a3b8; font-family:system-ui; font-weight:800; margin-bottom:6px;">GOLD (coverage target)</div>
          <div style="color:#e5e7eb; font-family:ui-monospace; line-height:1.6; font-size:13px;">{gold_html}</div>
        </div>
        <div style="background:#0f172a; border:1px solid #1f2937; border-radius:12px; padding:10px;">
          <div style="color:#94a3b8; font-family:system-ui; font-weight:800; margin-bottom:6px;">XLM-R PRED spans</div>
          <div style="color:#e5e7eb; font-family:ui-monospace; line-height:1.6; font-size:13px;">{pred_html}</div>
        </div>
      </div>
    </div>
    """
    display(HTML(html))


In [ ]:
#XLMR_COL = "free_pred_spans__xlm"
#XLMR_COL = "free_pred_spans__bert_gr"
#XLMR_COL = "free_pred_spans__spacy_el"
XLMR_COL= "free_pred_llama"
GOLD_COL = "gold_spans_free_text"

groups = collect_xlmr_examples(
    df,
    gold_col=GOLD_COL,
    pred_col=XLMR_COL,
    text_col="free_text",
    doc_col="file",       # αν δεν έχεις τέτοια στήλη, βάλε None
    max_per_group=3
)

print("covered_not_exact:", len(groups["covered_not_exact"]))
print("exact:", len(groups["exact"]))
print("not_covered:", len(groups["not_covered"]))

for ex in groups["covered_not_exact"]:
    render_gold_vs_preds_snippet(df, ex, GOLD_COL, XLMR_COL, text_col="free_text", context=70)

for ex in groups["exact"]:
    render_gold_vs_preds_snippet(df, ex, GOLD_COL, XLMR_COL, text_col="free_text", context=70)

for ex in groups["not_covered"]:
    render_gold_vs_preds_snippet(df, ex, GOLD_COL, XLMR_COL, text_col="free_text", context=70)


covered_not_exact: 1
exact: 3
not_covered: 3


SUMMARIES

In [ ]:
import pandas as pd

def build_tau_tables_and_export(
    df,
    gold_col,
    model_pred_cols,              # dict: {"XLM-R": "free_pred_spans__xlm", ...}
    taus=(0.8, 0.9, 1.0),
    out_path="coverage_tables_by_tau.xlsx",
    decimals=3,                   # <- εδώ “φαίνεται όπως recall” (π.χ. 3 δεκαδικά)
    as_percent=False              # <- αν το βάλεις True θα γράφει % (π.χ. 81.2)
):
    tau_tables = {}

    for tau in taus:
        rows = []
        for model_name, pred_col in model_pred_cols.items():
            g = global_label_agnostic_coverage(df, gold_col, pred_col, tau=tau)

            rows.append({
                "model": model_name,
                "tp_pred": g["tp"],
                "fp": g["fp"],
                "tp_gold": g["tp_gold"],
                "fn": g["fn"],
                "precision": g["precision"],
                "recall": g["recall"],
                "f1": g["f1"],
            })

        tdf = pd.DataFrame(rows).sort_values("model")

        # formatting metrics
        metric_cols = ["precision", "recall", "f1"]
        if as_percent:
            tdf[metric_cols] = (tdf[metric_cols] * 100).round(1)   # π.χ. 81.2
        else:
            tdf[metric_cols] = tdf[metric_cols].round(decimals)    # π.χ. 0.812

        tau_tables[tau] = tdf

    # export to single Excel with 3 sheets
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for tau, tdf in tau_tables.items():
            sheet = f"tau_{str(tau).replace('.','_')}"  # tau_0_8, tau_0_9, tau_1_0
            tdf.to_excel(writer, sheet_name=sheet[:31], index=False)

            # freeze header row
            ws = writer.book[sheet[:31]]
            ws.freeze_panes = "A2"

    return tau_tables, out_path


# ---- ΧΡΗΣΗ ----
MODEL_PREDS = {
    "XLM-R":   "free_pred_spans__xlm",
    "spaCy-el":"free_pred_spans__spacy_el",
    "GR-BERT":"free_pred_spans__bert_gr",
    "Llama3":  "free_pred_llama",
}

tau_tables, out_path = build_tau_tables_and_export(
    df=df,
    gold_col="gold_spans_free_text",
    model_pred_cols=MODEL_PREDS,
    taus=(0.8, 0.9, 1.0),
    out_path="coverage_tables_by_tau.xlsx",
    decimals=3,       # precision/recall/f1 με ίδιο “look”
    as_percent=False  # κάν’το True αν θες να δείχνει 0–100
)

print("Saved:", out_path)
for tau, tdf in tau_tables.items():
    print("\n---", tau, "---")
    display(tdf)


Saved: coverage_tables_by_tau.xlsx

--- 0.8 ---


,model,tp_pred,fp,tp_gold,fn,precision,recall,f1
2,GR-BERT,41,271,39,11,0.131,0.78,0.225
3,Llama3,0,0,0,50,0.000,0.00,0.000
0,XLM-R,102,104,49,1,0.495,0.98,0.658
1,spaCy-el,48,353,26,24,0.120,0.52,0.195



--- 0.9 ---


,model,tp_pred,fp,tp_gold,fn,precision,recall,f1
2,GR-BERT,41,271,38,12,0.131,0.76,0.224
3,Llama3,0,0,0,50,0.000,0.00,0.000
0,XLM-R,102,104,39,11,0.495,0.78,0.606
1,spaCy-el,48,353,23,27,0.120,0.46,0.190



--- 1.0 ---


,model,tp_pred,fp,tp_gold,fn,precision,recall,f1
2,GR-BERT,41,271,26,24,0.131,0.52,0.210
3,Llama3,0,0,0,50,0.000,0.00,0.000
0,XLM-R,102,104,26,24,0.495,0.52,0.507
1,spaCy-el,48,353,23,27,0.120,0.46,0.190


In [ ]:
tau_tables, out_path = build_tau_tables_and_export(
    df=df,
    gold_col="gold_spans_free_text",
    model_pred_cols=MODEL_PREDS,
    taus=(0.8, 0.9, 1.0),
    out_path="/content/drive/MyDrive/Archimedes_Anonymization/coverage_tables_by_tau.xlsx",
    decimals=3,
    as_percent=False
)

print("Saved to Drive:", out_path)

Saved to Drive: /content/drive/MyDrive/Archimedes_Anonymization/coverage_tables_by_tau.xlsx


In [ ]:
from IPython.display import display, HTML
import re

def _word_window(text, center_s, center_e, words_before=6, words_after=6):
    """
    Βρίσκει όρια context με βάση λέξεις (όχι chars):
    επιστρέφει (cs, ce) indices στο text.
    """
    if not text:
        return (0, 0)

    # tokenize σε "words" + positions
    for_match = list(re.finditer(r"\S+", text))
    if not for_match:
        return (0, min(len(text), 1))

    # βρες word index που περιέχει center_s
    w_idx = 0
    for i, m in enumerate(for_match):
        if m.start() <= center_s < m.end():
            w_idx = i
            break
        if center_s >= m.end():
            w_idx = i

    start_idx = max(0, w_idx - words_before)
    end_idx = min(len(for_match) - 1, w_idx + words_after)

    cs = for_match[start_idx].start()
    ce = for_match[end_idx].end()

    return cs, ce

def _segments_within_window(gs, ge, pred_intervals, win_s, win_e):
    """
    Φτιάχνει segments για render ΜΟΝΟ μέσα στο [win_s, win_e],
    αλλά τα χρώματα εφαρμόζονται μόνο στο intersection με gold [gs,ge]
    (πράσινο=covered, κόκκινο=missed).
    """
    # window text baseline: no color
    # within gold intersection, split to covered/missed using union of preds (clipped to gold)
    segs_gold = gold_segments_covered_vs_missed(gs, ge, pred_intervals)  # (s,e,is_cov) within gold
    # clip segs_gold to window
    segs_gold = [(max(s,win_s), min(e,win_e), is_cov) for s,e,is_cov in segs_gold if max(s,win_s) < min(e,win_e)]

    # build final segments over the window
    segs = []
    cur = win_s

    # sort gold segments
    segs_gold.sort(key=lambda x: x[0])

    for s,e,is_cov in segs_gold:
        if cur < s:
            segs.append((cur, s, "plain"))
        segs.append((s, e, "covered" if is_cov else "missed"))
        cur = e

    if cur < win_e:
        segs.append((cur, win_e, "plain"))

    return segs

def print_all_FNs_compact_html(
    df,
    gold_col,
    pred_col,
    text_col="free_text",
    doc_col="file",
    tau=0.8,
    max_examples=None,
    min_overlap_chars=1,
    words_before=6,
    words_after=6
):
    def esc(x):
        return (x.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;"))

    blocks = []
    shown = 0

    for idx, r in df.iterrows():
        text = r.get(text_col, "") or ""
        gold_spans = r.get(gold_col) or []
        pred_spans = r.get(pred_col) or []
        pred_intervals = [(s, e) for s, e, _ in pred_spans]

        file_name = r.get(doc_col, f"ROW {idx}")

        for gs, ge, glab in gold_spans:
            glen = max(0, ge - gs)
            if glen == 0:
                continue

            covered = _union_covered_len((gs, ge), pred_intervals)
            cov = covered / glen if glen else 0.0
            if cov >= tau:
                continue  # όχι FN

            # GOLD span text
            gold_txt = text[gs:ge].replace("\n"," ")

            # overlapping predicted spans (exact outputs)
            overlaps = []
            for ps, pe, plab in pred_spans:
                inter = _intersection_len((ps, pe), (gs, ge))
                if inter >= min_overlap_chars:
                    ptxt = text[ps:pe].replace("\n"," ")
                    overlaps.append((plab, inter, ptxt))
            overlaps.sort(key=lambda x: x[1], reverse=True)

            # context window (λίγες λέξεις πριν/μετά)
            win_s, win_e = _word_window(text, gs, ge, words_before=words_before, words_after=words_after)
            segs = _segments_within_window(gs, ge, pred_intervals, win_s, win_e)

            # render colored context line
            ctx_parts = []
            for s,e,typ in segs:
                chunk = esc(text[s:e].replace("\n"," "))
                if typ == "covered":
                    ctx_parts.append(f'<span style="color:#22c55e; font-weight:800;">{chunk}</span>')
                elif typ == "missed":
                    ctx_parts.append(f'<span style="color:#ef4444; font-weight:800;">{chunk}</span>')
                else:
                    ctx_parts.append(f'<span style="color:#e5e7eb;">{chunk}</span>')

            # add ellipses if window is clipped
            prefix = "… " if win_s > 0 else ""
            suffix = " …" if win_e < len(text) else ""

            # build block
            html = []
            html.append(
                f'<div style="font-family:monospace; background:#0f172a; color:#e5e7eb; '
                f'padding:12px; margin-bottom:12px; border-radius:8px;">'
            )

            html.append(f'<div style="color:#38bdf8; font-weight:900;">FILE: {esc(str(file_name))}</div>')
            html.append(
                f'<div>'
                f'<span style="color:#ef4444; font-weight:900;">FN GOLD</span> | '
                f'label=<span style="color:#c084fc; font-weight:900;">{esc(glab)}</span> | '
                f'coverage=<span style="color:#f59e0b; font-weight:900;">{cov:.2f}</span> '
                f'<span style="color:#94a3b8;">(tau={tau})</span>'
                f'</div>'
            )

            # GOLD green
            html.append(
                f'<div style="margin-left:16px; margin-top:8px;">'
                f'<span style="color:#94a3b8; font-weight:900;">GOLD:</span> '
                f'<span style="color:#22c55e; font-weight:900;">\'{esc(gold_txt)}\'</span>'
                f'</div>'
            )

            # PRED exact no color
            html.append(
                f'<div style="margin-left:16px; margin-top:6px;">'
                f'<span style="color:#94a3b8; font-weight:900;">PRED (exact):</span> '
            )
            if overlaps:
                # χωρίς χρώμα (λευκό/γκρι)
                html.append('<div style="margin-left:32px; color:#e5e7eb;">')
                for plab, inter, ptxt in overlaps:
                    html.append(f"- {esc(plab)} | overlap_chars={inter} '{esc(ptxt)}'<br>")
                html.append("</div>")
            else:
                html.append(f"<span style='color:#e5e7eb;'>NONE (no overlap)</span>")
            html.append("</div>")

            # Context colored line
            html.append(
                f'<div style="margin-left:16px; margin-top:6px;">'
                f'<span style="color:#94a3b8; font-weight:900;">CONTEXT (green=covered, red=missed):</span><br>'
                f'<span style="color:#94a3b8;">{esc(prefix)}</span>'
                f'<span>\'{"" .join(ctx_parts)}\'</span>'
                f'<span style="color:#94a3b8;">{esc(suffix)}</span>'
                f'</div>'
            )

            html.append('</div>')
            blocks.append("".join(html))

            shown += 1
            if max_examples is not None and shown >= max_examples:
                display(HTML("".join(blocks)))
                return

    display(HTML("".join(blocks)))


In [ ]:
print_all_FNs_compact_html(
    df,
    gold_col="gold_spans_free_text",
    #pred_col="free_pred_llama",
    pred_col="free_pred_spans__xlm",
    #pred_col="free_pred_spans__spacy_el",
    #pred_col="free_pred_spans__bert_gr",
    text_col="free_text",
    doc_col="file",
    tau=1,
    max_examples=None,   # όλα τα FN
    words_before=6,
    words_after=6
)


In [ ]:
def gold_segments_covered_vs_missed(gold_s, gold_e, pred_intervals):
    """
    Input: a gold span (s,e), and a list of predicted intervals (s,e).
    Output: a list of segments (s,e,is_covered) within the gold span,
            where `is_covered` means that segment is covered by any pred_interval.
    """
    gs, ge = gold_s, gold_e
    segs = []
    if gs >= ge:
        return []

    # filter/clip pred_intervals to gold_span
    clipped_preds = []
    for ps, pe in pred_intervals:
        s2 = max(gs, ps)
        e2 = min(ge, pe)
        if s2 < e2:
            clipped_preds.append((s2, e2))

    if not clipped_preds:
        return [(gs, ge, False)]  # no predictions = all missed

    # merge overlapping clipped_preds to get the union of covered regions
    clipped_preds.sort()
    merged_covered = []
    cur_s, cur_e = clipped_preds[0]
    for s, e in clipped_preds[1:]:
        if s <= cur_e:
            cur_e = max(cur_e, e)
        else:
            merged_covered.append((cur_s, cur_e))
            cur_s, cur_e = s, e
    merged_covered.append((cur_s, cur_e))

    # now create segments: iterate through gold span and mark covered/missed
    cur = gs
    for cov_s, cov_e in merged_covered:
        if cur < cov_s:
            segs.append((cur, cov_s, False)) # missed segment
        segs.append((cov_s, cov_e, True))  # covered segment
        cur = cov_e
    if cur < ge:
        segs.append((cur, ge, False))  # remaining missed segment

    return segs

In [ ]:
from IPython.display import display, HTML

def clip_spans_to_window(spans, win_s, win_e):
    out = []
    for s,e,l in (spans or []):
        s2 = max(s, win_s)
        e2 = min(e, win_e)
        if s2 < e2:
            out.append((s2 - win_s, e2 - win_s, l))  # shift to snippet coords
    return out

def render_spans_html_card(title, text, spans, colors=PASTEL_COLORS):
    spans = sorted(spans, key=lambda x: x[0])
    html = f"""
    <div style="margin:10px 0; padding:10px; border:1px solid #ddd; border-radius:10px;">
      <div style="font-weight:700; margin-bottom:6px;">{title}</div>
      <div style="font-family: monospace; line-height: 1.5;">
    """
    i = 0
    for s,e,lab in spans:
        html += text[i:s]
        color = colors.get(lab, "#f0f0f0")
        html += (
            f'<span style="background:{color}; padding:2px 4px; border-radius:4px; white-space:pre-wrap;">'
            f'{text[s:e]}'
            f'<sup style="font-size:0.7em; margin-left:4px; color:#666;">{lab}</sup>'
            f'</span>'
        )
        i = e
    html += text[i:]
    html += "</div></div>"
    display(HTML(html.replace("\n","<br>")))

def show_models_on_same_snippet(row, text_col, model_cols, center=None, window_chars=400):
    """
    model_cols: dict {"XLM-R": "free_pred_spans__xlm", ...}
    center: None ή int index (char) ή tuple(gs,ge) (span)
    """
    text = row[text_col]
    n = len(text)

    if center is None:
        # default: αρχή του κειμένου
        mid = min(n//6, n-1)
    elif isinstance(center, int):
        mid = max(0, min(center, n-1))
    else:
        gs, ge = center
        mid = (gs + ge)//2

    win_s = max(0, mid - window_chars//2)
    win_e = min(n, win_s + window_chars)
    snippet = text[win_s:win_e]

    # δείξε και ένα μικρό header με το εύρος
    display(HTML(f"<div style='color:#666; margin:6px 0;'>Snippet chars: [{win_s}, {win_e}]</div>"))

    for model_name, col in model_cols.items():
        spans = row[col]
        clipped = clip_spans_to_window(spans, win_s, win_e)
        render_spans_html_card(model_name, snippet, clipped)


In [ ]:
i = 13
row = df.iloc[i]

MODELS = {
    "XLM-R": "free_pred_spans__xlm",
    "spaCy-el": "free_pred_spans__spacy_el",
    "GR-BERT": "free_pred_spans__bert_gr",
    "Llama3": "free_pred_llama",
}

# Κέντρο: π.χ. στο πρώτο gold span ή σε index
gs,ge,_ = row["gold_spans_free_text"][0]
show_models_on_same_snippet(row, "free_text", MODELS, center=(gs,ge), window_chars=450)


In [ ]:
from IPython.display import display, HTML

# -----------------------
# COLORS (same as yours)
# -----------------------
PASTEL_COLORS = {
    "person_name":   "#c084fc",
    "staff_name":    "#60a5fa",
    "hospital_name": "#4ade80",
    "location":      "#60a5fa",
    "phone":         "#f87171",
    "address":       "#f7f7f7",
    "postal_city":   "#93c5fd",
    "patient_id":    "#f6f6f6",
    "misc":          "#93c5fd",
}

# -----------------------
# Span rendering (inline HTML) + dark theme cards
# -----------------------
def clip_spans_to_window(spans, win_s, win_e):
    """
    Clip spans to [win_s, win_e] in the ORIGINAL text coords,
    then shift them to snippet coords (0..snippet_len).
    """
    out = []
    for s, e, l in (spans or []):
        s2 = max(s, win_s)
        e2 = min(e, win_e)
        if s2 < e2:
            out.append((s2 - win_s, e2 - win_s, l))
    return out

def render_spans_html_inline(text, spans, colors=PASTEL_COLORS):
    """
    Return HTML string with highlighted spans (no display()).
    """
    spans = sorted(spans, key=lambda x: x[0])
    html = ""
    i = 0

    for s, e, lab in spans:
        html += text[i:s]
        color = colors.get(lab, "#f0f0f0")
        html += (
            f'<span style="background:{color}; '
            f'padding:2px 4px; '
            f'border-radius:4px; '
            f'white-space:pre-wrap;">'
            f'{text[s:e]}'
            f'<sup style="font-size:0.7em; margin-left:4px; color:#94a3b8;">{lab}</sup>'
            f'</span>'
        )
        i = e

    html += text[i:]
    return html.replace("\n", "<br>")

def choose_window(text, center=None, window_chars=450):
    """
    Choose a snippet window [win_s, win_e] around:
      - None: near the beginning
      - int: a character index
      - (gs,ge): a span
    """
    n = len(text)
    if n == 0:
        return 0, 0

    if center is None:
        mid = min(max(0, n // 6), n - 1)
    elif isinstance(center, int):
        mid = max(0, min(center, n - 1))
    else:
        gs, ge = center
        mid = (gs + ge) // 2
        mid = max(0, min(mid, n - 1))

    win_s = max(0, mid - window_chars // 2)
    win_e = min(n, win_s + window_chars)
    return win_s, win_e

def make_model_card_html(title, snippet, clipped_spans):
    """
    Dark-theme card, slide-friendly.
    """
    body = render_spans_html_inline(snippet, clipped_spans)
    return f"""
    <div style="
        padding:12px;
        border:1px solid #1f2937;
        border-radius:14px;
        background:#0f172a;
        box-shadow: 0 4px 10px rgba(0,0,0,0.35);
    ">
      <div style="
          font-weight:800;
          margin-bottom:8px;
          font-family:system-ui;
          color:#e5e7eb;
      ">
        {title}
      </div>
      <div style="
          font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;
          line-height:1.6;
          font-size:13.5px;
          color:#e5e7eb;
      ">
        {body}
      </div>
    </div>
    """

def _snippet_header_html(win_s, win_e):
    return f"""
    <div style="
        color:#9ca3af;
        margin:8px 0;
        font-family:system-ui;
        font-size:13px;
    ">
      Snippet chars: [{win_s}, {win_e}]
    </div>
    """

# -----------------------
# FORMAT A: stacked (one below the other)
# -----------------------
def show_models_stacked(
    row,
    text_col,
    model_cols,          # dict: {"XLM-R": "free_pred_spans__xlm", ...}
    center=None,
    window_chars=450,
    show_snippet_range=True
):
    text = row[text_col]
    win_s, win_e = choose_window(text, center=center, window_chars=window_chars)
    snippet = text[win_s:win_e]

    cards = []
    for model_name, col in model_cols.items():
        spans = row[col]
        clipped = clip_spans_to_window(spans, win_s, win_e)
        cards.append(make_model_card_html(model_name, snippet, clipped))

    header = _snippet_header_html(win_s, win_e) if show_snippet_range else ""
    html = (
        "<div style='background:#0b1220; padding:12px; border-radius:16px;'>"
        + header
        + "<div style='display:flex; flex-direction:column; gap:10px;'>"
        + "".join(cards)
        + "</div></div>"
    )
    display(HTML(html))

# -----------------------
# FORMAT B: grid / side-by-side (e.g., 2x2)
# -----------------------
def show_models_grid(
    row,
    text_col,
    model_cols,          # dict
    center=None,
    window_chars=450,
    n_cols=2,
    show_snippet_range=True
):
    text = row[text_col]
    win_s, win_e = choose_window(text, center=center, window_chars=window_chars)
    snippet = text[win_s:win_e]

    cards = []
    for model_name, col in model_cols.items():
        spans = row[col]
        clipped = clip_spans_to_window(spans, win_s, win_e)
        cards.append(make_model_card_html(model_name, snippet, clipped))

    header = _snippet_header_html(win_s, win_e) if show_snippet_range else ""
    html = f"""
    <div style="background:#0b1220; padding:12px; border-radius:16px;">
      {header}
      <div style="display:grid; grid-template-columns: repeat({n_cols}, minmax(0, 1fr)); gap:10px;">
        {''.join(cards)}
      </div>
    </div>
    """
    display(HTML(html))


# -----------------------
# Example usage
# -----------------------
# i = 13
# row = df.iloc[i]
#
# MODELS = {
#     "XLM-R": "free_pred_spans__xlm",
#     "spaCy-el": "free_pred_spans__spacy_el",
#     "GR-BERT": "free_pred_spans__bert_gr",
#     "Llama3": "free_pred_llama",
# }
#
# # center can be: None, an int char index, or (gs,ge) from a gold span
# gs, ge, _ = row["gold_spans_free_text"][0]
#
# # Format A: stacked
# show_models_stacked(row, "free_text", MODELS, center=(gs, ge), window_chars=450)
#
# # Format B: 2x2 grid
# show_models_grid(row, "free_text", MODELS, center=(gs, ge), window_chars=450, n_cols=2)


In [ ]:
i = 13
row = df.iloc[i]

MODELS = {
    "XLM-R": "free_pred_spans__xlm",
    "spaCy-el": "free_pred_spans__spacy_el",
    "GR-BERT": "free_pred_spans__bert_gr",
    "Llama3": "free_pred_llama",
}

# Κέντρο: μπορείς να βάλεις gold span (για να “πέσει” σε ενδιαφέρον σημείο)
gs, ge, _ = row["gold_spans_free_text"][0]

# Format A: stacked (κάτω-κάτω)
#show_models_stacked(row, "free_text", MODELS, center=(gs, ge), window_chars=450)

# Format B: grid 2x2 (side-by-side)
show_models_grid(row, "free_text", MODELS, center=(gs, ge), window_chars=450, n_cols=2)


In [ ]:
from IPython.display import display, HTML

# -----------------------
# COLORS
# -----------------------
PASTEL_COLORS = {
    "person_name":   "#c084fc",
    "staff_name":    "#60a5fa",
    "hospital_name": "#4ade80",
    "location":      "#60a5fa",
    "phone":         "#f87171",
    "address":       "#f7f7f7",
    "postal_city":   "#93c5fd",
    "patient_id":    "#f6f6f6",
    "misc":          "#93c5fd",
}

# Gold visualization color (single color for all gold spans)
GOLD_HIGHLIGHT_COLOR = "#fbbf24"  # yellow

# -----------------------
# Helpers
# -----------------------
def clip_spans_to_window(spans, win_s, win_e):
    out = []
    for s, e, l in (spans or []):
        s2 = max(s, win_s)
        e2 = min(e, win_e)
        if s2 < e2:
            out.append((s2 - win_s, e2 - win_s, l))  # shift
    return out

def render_spans_html_inline(text, spans, colors):
    spans = sorted(spans, key=lambda x: x[0])
    html = ""
    i = 0
    for s, e, lab in spans:
        html += text[i:s]
        color = colors.get(lab, "#f0f0f0")
        html += (
            f'<span style="background:{color}; padding:2px 4px; border-radius:4px; white-space:pre-wrap;">'
            f'{text[s:e]}'
            f'<sup style="font-size:0.7em; margin-left:4px; color:#94a3b8;">{lab}</sup>'
            f'</span>'
        )
        i = e
    html += text[i:]
    return html.replace("\n", "<br>")

def choose_window(text, center=None, window_chars=450):
    n = len(text)
    if n == 0:
        return 0, 0

    if center is None:
        mid = min(max(0, n // 6), n - 1)
    elif isinstance(center, int):
        mid = max(0, min(center, n - 1))
    else:
        gs, ge = center
        mid = (gs + ge) // 2
        mid = max(0, min(mid, n - 1))

    win_s = max(0, mid - window_chars // 2)
    win_e = min(n, win_s + window_chars)
    return win_s, win_e

def make_model_card_html(title, snippet, clipped_spans, colors, badge=None):
    body = render_spans_html_inline(snippet, clipped_spans, colors=colors)

    badge_html = ""
    if badge:
        badge_html = f"""
        <span style="
            margin-left:8px;
            font-size:12px;
            padding:2px 8px;
            border-radius:999px;
            background:#111827;
            border:1px solid #1f2937;
            color:#93c5fd;
            font-weight:700;
        ">{badge}</span>
        """

    return f"""
    <div style="
        padding:12px;
        border:1px solid #1f2937;
        border-radius:14px;
        background:#0f172a;
        box-shadow:0 4px 10px rgba(0,0,0,0.35);
        min-height: 190px;
    ">
      <div style="display:flex; align-items:center; justify-content:space-between; gap:10px;">
        <div style="font-weight:900; font-family:system-ui; color:#e5e7eb;">
          {title}{badge_html}
        </div>
      </div>
      <div style="
          margin-top:8px;
          font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;
          line-height:1.6;
          font-size:13.5px;
          color:#e5e7eb;
      ">
        {body}
      </div>
    </div>
    """

def _snippet_header_html(win_s, win_e):
    return f"""
    <div style="color:#9ca3af; margin:8px 0; font-family:system-ui; font-size:13px;">
      Snippet chars: [{win_s}, {win_e}]
    </div>
    """

# -----------------------
# Main: 2x2 grid with GOLD (top-left) + 3 models (no spaCy)
# -----------------------
def show_gold_plus_models_grid(
    row,
    text_col,
    gold_col,
    model_cols,              # dict with 3 models ONLY (XLM-R, GR-BERT, Llama3)
    center=None,
    window_chars=450,
    show_snippet_range=True
):
    text = row[text_col]
    win_s, win_e = choose_window(text, center=center, window_chars=window_chars)
    snippet = text[win_s:win_e]

    # GOLD spans: convert to single "gold" label so we can color them all yellow
    gold_spans = row[gold_col] or []
    gold_clipped = clip_spans_to_window(gold_spans, win_s, win_e)
    gold_as_single_label = [(s, e, "GOLD") for (s, e, _) in gold_clipped]
    gold_colors = {"GOLD": GOLD_HIGHLIGHT_COLOR}

    cards = []
    # Top-left: GOLD
    cards.append(make_model_card_html("GOLD", snippet, gold_as_single_label, colors=gold_colors, badge="yellow"))

    # The 3 models
    for model_name, col in model_cols.items():
        spans = row[col]
        clipped = clip_spans_to_window(spans, win_s, win_e)
        cards.append(make_model_card_html(model_name, snippet, clipped, colors=PASTEL_COLORS))

    header = _snippet_header_html(win_s, win_e) if show_snippet_range else ""
    html = f"""
    <div style="background:#0b1220; padding:12px; border-radius:16px;">
      {header}
      <div style="display:grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap:10px;">
        {''.join(cards)}
      </div>
    </div>
    """
    display(HTML(html))



In [ ]:

i = 19
row = df.iloc[i]
MODELS_3 = {
     "XLM-R": "free_pred_spans__xlm",
     "GR-BERT": "free_pred_spans__bert_gr",
     "Llama3": "free_pred_llama",
 }

#center can be None, int, or (gs,ge) from a gold span
gs, ge, _ = row["gold_spans_free_text"][0]
show_gold_plus_models_grid(
     row,
     text_col="free_text",
     gold_col="gold_spans_free_text",
     model_cols=MODELS_3,
     center=(gs, ge),
     window_chars=1050
 )


In [ ]:
from IPython.display import display, HTML

# -----------------------
# COLORS
# -----------------------
PASTEL_COLORS = {
    "person_name":   "#c084fc",
    "staff_name":    "#60a5fa",
    "hospital_name": "#4ade80",
    "location":      "#60a5fa",
    "phone":         "#f87171",
    "address":       "#f7f7f7",
    "postal_city":   "#93c5fd",
    "patient_id":    "#f6f6f6",
    "misc":          "#93c5fd",
}

# Gold visualization color (single color for all gold spans)
GOLD_HIGHLIGHT_COLOR = "#fbbf24"  # yellow

# -----------------------
# Helpers
# -----------------------
def clip_spans_to_window(spans, win_s, win_e):
    """
    Clip spans to [win_s, win_e] in the ORIGINAL text coords,
    then shift them to snippet coords (0..snippet_len).
    """
    out = []
    for s, e, l in (spans or []):
        s2 = max(s, win_s)
        e2 = min(e, win_e)
        if s2 < e2:
            out.append((s2 - win_s, e2 - win_s, l))  # shift
    return out

def render_spans_html_inline(text, spans, colors, show_labels=False):
    """
    Return HTML string with highlighted spans (no display()).
    show_labels=False => does NOT show entity label text (hospital_name, etc.)
    """
    spans = sorted(spans, key=lambda x: x[0])
    html = ""
    i = 0

    for s, e, lab in spans:
        html += text[i:s]
        color = colors.get(lab, "#f0f0f0")

        sup = ""
        if show_labels:
            # muted label (if you ever want it back)
            sup = f'<sup style="font-size:0.65em; margin-left:4px; color:#6b7280;">{lab}</sup>'

        html += (
            f'<span style="background:{color}; padding:2px 4px; border-radius:4px; white-space:pre-wrap;">'
            f'{text[s:e]}'
            f'{sup}'
            f'</span>'
        )
        i = e

    html += text[i:]
    return html.replace("\n", "<br>")

def choose_window(text, center=None, window_chars=450):
    """
    Choose a snippet window [win_s, win_e] around:
      - None: near the beginning
      - int: a character index
      - (gs,ge): a span
    """
    n = len(text)
    if n == 0:
        return 0, 0

    if center is None:
        mid = min(max(0, n // 6), n - 1)
    elif isinstance(center, int):
        mid = max(0, min(center, n - 1))
    else:
        gs, ge = center
        mid = (gs + ge) // 2
        mid = max(0, min(mid, n - 1))

    win_s = max(0, mid - window_chars // 2)
    win_e = min(n, win_s + window_chars)
    return win_s, win_e

def make_model_card_html(title, snippet, clipped_spans, colors, badge=None, show_labels=False):
    """
    Dark-theme card, slide-friendly.
    """
    body = render_spans_html_inline(snippet, clipped_spans, colors=colors, show_labels=show_labels)

    badge_html = ""
    if badge:
        badge_html = f"""
        <span style="
            margin-left:8px;
            font-size:12px;
            padding:2px 8px;
            border-radius:999px;
            background:#111827;
            border:1px solid #1f2937;
            color:#93c5fd;
            font-weight:700;
        ">{badge}</span>
        """

    return f"""
    <div style="
        padding:12px;
        border:1px solid #1f2937;
        border-radius:14px;
        background:#0f172a;
        box-shadow:0 4px 10px rgba(0,0,0,0.35);
        min-height: 190px;
    ">
      <div style="display:flex; align-items:center; justify-content:space-between; gap:10px;">
        <div style="font-weight:900; font-family:system-ui; color:#e5e7eb;">
          {title}{badge_html}
        </div>
      </div>
      <div style="
          margin-top:8px;
          font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;
          line-height:1.6;
          font-size:13.5px;
          color:#e5e7eb;
      ">
        {body}
      </div>
    </div>
    """

def _snippet_header_html(win_s, win_e):
    return f"""
    <div style="color:#9ca3af; margin:8px 0; font-family:system-ui; font-size:13px;">
      Snippet chars: [{win_s}, {win_e}]
    </div>
    """

# -----------------------
# Main: 2x2 grid with GOLD (top-left) + 3 models (no spaCy)
# -----------------------
def show_gold_plus_models_grid(
    row,
    text_col,
    gold_col,
    model_cols,              # dict with 3 models ONLY (XLM-R, GR-BERT, Llama3)
    center=None,
    window_chars=450,
    show_snippet_range=True,
    show_labels=False        # False => no "hospital_name" sup labels anywhere
):
    text = row[text_col]
    win_s, win_e = choose_window(text, center=center, window_chars=window_chars)
    snippet = text[win_s:win_e]

    # GOLD spans: convert to single "GOLD" label so we can color them all yellow
    gold_spans = row[gold_col] or []
    gold_clipped = clip_spans_to_window(gold_spans, win_s, win_e)
    gold_as_single_label = [(s, e, "GOLD") for (s, e, _) in gold_clipped]
    gold_colors = {"GOLD": GOLD_HIGHLIGHT_COLOR}

    cards = []
    # Top-left: GOLD
    cards.append(
        make_model_card_html(
            "GOLD",
            snippet,
            gold_as_single_label,
            colors=gold_colors,
            badge="yellow",
            show_labels=show_labels
        )
    )

    # The 3 models
    for model_name, col in model_cols.items():
        spans = row[col]
        clipped = clip_spans_to_window(spans, win_s, win_e)
        cards.append(
            make_model_card_html(
                model_name,
                snippet,
                clipped,
                colors=PASTEL_COLORS,
                show_labels=show_labels
            )
        )

    header = _snippet_header_html(win_s, win_e) if show_snippet_range else ""
    html = f"""
    <div style="background:#0b1220; padding:12px; border-radius:16px;">
      {header}
      <div style="display:grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap:10px;">
        {''.join(cards)}
      </div>
    </div>
    """
    display(HTML(html))


# -----------------


In [ ]:
i = 10
row = df.iloc[i]
MODELS_3 = {
     "XLM-R": "free_pred_spans__xlm",
     "GR-BERT": "free_pred_spans__bert_gr",
     "Llama3": "free_pred_llama",
 }
#center can be None, int, or (gs,ge) from a gold span
gs, ge, _ = row["gold_spans_free_text"][0]
show_gold_plus_models_grid(
     row,
     text_col="free_text",
     gold_col="gold_spans_free_text",
     model_cols=MODELS_3,
     center=(gs, ge),
     window_chars=450
 )


In [ ]:
df

,file,raw_text,synthetic_text,synthetic_map,clean_text,gold_spans,template_text,free_text,gold_spans_template,gold_spans_free_text,pred_spans_template,free_pred_spans__spacy_el,free_pred_spans__xlm,free_pred_spans__bert_gr,free_pred_llama
0,Ασθ1.docx,##############################################...,##############################################...,"{'@74930277': {'type': 'patient_id', 'value': ...",##############################################...,"[(326, 334, patient_id), (403, 416, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ – ΠΟΡΕΙΑ ΝΟΣΟΥ\nΑτομι...,"[(326, 334, patient_id), (403, 416, staff_name...","[(298, 321, hospital_name), (366, 389, hospita...","[(326, 334, patient_id), (403, 416, staff_name...","[(298, 307, hospital_name), (366, 375, hospita...","[(297, 307, hospital_name), (309, 320, locatio...","[(243, 245, hospital_name), (298, 321, hospita...","[(298, 321, hospital_name), (366, 389, hospita..."
1,Ασθ2.docx,##############################################...,##############################################...,"{'@745625100': {'type': 'patient_id', 'value':...",##############################################...,"[(325, 333, patient_id), (414, 441, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\n\nΠρογραμματισμένη ε...,"[(325, 333, patient_id), (414, 441, staff_name...","[(60, 90, hospital_name), (2897, 2927, hospita...","[(325, 333, patient_id), (414, 441, staff_name...","[(60, 75, person_name), (1727, 1735, hospital_...","[(55, 75, hospital_name), (77, 89, location), ...","[(56, 58, hospital_name), (58, 89, hospital_na...","[(60, 75, hospital_name), (77, 89, person_name..."
2,Ασθ3.docx,##############################################...,##############################################...,"{'@84726155': {'type': 'patient_id', 'value': ...",##############################################...,"[(326, 334, patient_id), (415, 434, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\nΑιτία εισόδου: Ασθεν...,"[(326, 334, patient_id), (415, 434, staff_name...","[(792, 822, hospital_name)]","[(326, 334, patient_id), (415, 434, staff_name...","[(25, 30, location), (526, 530, hospital_name)...","[(791, 793, hospital_name), (793, 807, locatio...","[(84, 106, hospital_name), (792, 822, hospital...","[(654, 661, person_name), (792, 822, hospital_..."
3,Ασθ4.docx,##############################################...,##############################################...,"{'@9832492': {'type': 'patient_id', 'value': '...",##############################################...,"[(325, 333, patient_id), (414, 427, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\nΑΙΤΙΑ ΕΙΣΟΔΟΥ:\nΟΠΟ-...,"[(325, 333, patient_id), (414, 427, staff_name...","[(953, 965, hospital_name)]","[(325, 333, patient_id), (414, 427, staff_name...","[(31, 38, hospital_name), (48, 68, person_name...","[(952, 957, location), (957, 965, location), (...","[(40, 47, hospital_name), (113, 114, hospital_...","[(850, 854, location), (953, 965, location), (..."
4,Ασθ5.docx,##############################################...,##############################################...,"{'@0912483': {'type': 'patient_id', 'value': '...",##############################################...,"[(326, 334, patient_id), (415, 436, staff_name...",##############################################...,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\nΑιτία εισόδου :Χειρο...,"[(326, 334, patient_id), (415, 436, staff_name...",[],"[(326, 334, patient_id), (415, 436, staff_name...","[(25, 30, location), (300, 310, hospital_name)...",[],"[(349, 351, hospital_name), (666, 668, hospita...","[(571, 581, location), (2726, 2731, person_name)]"
5,Ασθ6.docx,##############################################...,##############################################...,"{'@98437592': {'type': 'patient_id', 'value': ...",##############################################...,"[(325, 333, patient_id), (414, 431, st

In [ ]:
print(df['raw_text'][13])
print(df['clean_text'][13])

################################################################################
# FILE: Ασθ14.docx
################################################################################

===== word/document.xml =====

ΥΠΟΥΡΓΕΙΟ ΥΓΕΙΑΣ ΚΑΙ ΚΟΙΝΩΝΙΚΗΣ ΑΛΛΗΛΕΓΓΥΗΣ
Ι4

ΠΑΝΕΠΙΣΤΗΜΙΑΚΟ ΓΕΝΙΚΟ 
471

ΝΟΣΟΚΟΜΕΙΟ ΙΩΑΝΝΙΝΩΝ
Αρ. Μητρ. Ασθ.:
@9832742



Τμήμα/Κλινική:
Μονάδα Εντατικής Θεραπείας

Ομάδα Αίματος Rh:




Διευθυντής:
@βωκδωςσ

HBsAg:


Το ΛΕΥΚΟ  αντίγραφο στον ασθενή  - το ΓΑΛΑΖΙΟ στον ιατρικό φάκελοΕΝΗΜΕΡΩΤΙΚΟ ΣΗΜΕΙΩΜΑ
Το ΛΕΥΚΟ  αντίγραφο στον ασθενή  - το ΓΑΛΑΖΙΟ στον ιατρικό φάκελο
Το ενημερωτικό σημείωμα να το έχετε μαζί σας κάθε φορά που θα επισκέπτεστε τον ιατρό

ΣΤΟΙΧΕΙΑ ΑΣΘΕΝΟΥΣ

Επώνυμο:
@νωκλδσλ
Όνομα:
@ποφςεξφν
Ηλικία:
77
Διεύθυνση:

Τ.Κ. – Πόλη:

Τηλέφωνο:

Ημ/νία Εισόδου:
10/09/2024
Ημ/νία Εξόδου:
01/12/2024

ΑΙΤΙΑ ΕΙΣΟΔΟΥ - ΙΣΤΟΡΙΚΟ

 Αιτία εισόδου: Ασθενής ο οποίος νοσηλευόταν στο ΓΝ @βκωξσδβω από 11/06/2024, διεκομίσθη κατόπιν συνεννόησης στη ΜΕΘ του ΠΓΝΙ στις 10/09/2024 λόγω

In [ ]:
from IPython.display import display, HTML

i = 16
raw_text = df["raw_text"][i]
clean_text = df["clean_text"][i]

def esc(x):
    return (x.replace("&","&amp;")
             .replace("<","&lt;")
             .replace(">","&gt;"))

html = f"""
<div style="display:grid; grid-template-columns: 1fr 1fr; gap:20px;
            background:#0b1220; padding:16px; border-radius:16px;">

  <div>
    <div style="font-family:system-ui; font-weight:900; color:#94a3b8; margin-bottom:6px;">
      BEFORE (anonymized)
    </div>
    <pre style="
        background:#0f172a;
        color:#e5e7eb;
        padding:12px;
        border-radius:12px;
        font-size:13px;
        line-height:1.5;
        white-space:pre-wrap;
        word-break:break-word;
    ">
{esc(raw_text)}
    </pre>
  </div>

  <div>
    <div style="font-family:system-ui; font-weight:900; color:#94a3b8; margin-bottom:6px;">
      AFTER (synthetic)
    </div>
    <pre style="
        background:#0f172a;
        color:#e5e7eb;
        padding:12px;
        border-radius:12px;
        font-size:13px;
        line-height:1.5;
        white-space:pre-wrap;
        word-break:break-word;
    ">
{esc(clean_text)}
    </pre>
  </div>

</div>
"""

display(HTML(html))


In [ ]:
from IPython.display import display, HTML

def _first_diff_idx(a: str, b: str) -> int:
    n = min(len(a), len(b))
    for i in range(n):
        if a[i] != b[i]:
            return i
    return n

def _common_suffix_len(a: str, b: str, max_check=None) -> int:
    # length of common suffix
    if max_check is None:
        max_check = min(len(a), len(b))
    k = 0
    while k < max_check and a[-1-k] == b[-1-k]:
        k += 1
    return k

def render_anonymized_to_synthetic_arrows(
    raw_text: str,
    clean_text: str,
    context_chars=35,
    title="Anonymized → Synthetic (example)"
):
    raw_text = raw_text or ""
    clean_text = clean_text or ""

    def esc(x):
        return (x.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;"))

    # Find changed region approximately using first diff + common suffix
    i0 = _first_diff_idx(raw_text, clean_text)
    suf = _common_suffix_len(raw_text[i0:], clean_text[i0:])  # common suffix after i0
    # define change slices
    raw_end = max(i0, len(raw_text) - suf)
    clean_end = max(i0, len(clean_text) - suf)

    # context window around the change start/end
    raw_win_s = max(0, i0 - context_chars)
    raw_win_e = min(len(raw_text), raw_end + context_chars)
    clean_win_s = max(0, i0 - context_chars)
    clean_win_e = min(len(clean_text), clean_end + context_chars)

    raw_prefix = "… " if raw_win_s > 0 else ""
    raw_suffix = " …" if raw_win_e < len(raw_text) else ""
    clean_prefix = "… " if clean_win_s > 0 else ""
    clean_suffix = " …" if clean_win_e < len(clean_text) else ""

    # Build BEFORE line: highlight changed region in yellow
    raw_a = esc(raw_text[raw_win_s:i0])
    raw_b = esc(raw_text[i0:raw_end])
    raw_c = esc(raw_text[raw_end:raw_win_e])

    # Build AFTER line: highlight changed region in green
    clean_a = esc(clean_text[clean_win_s:i0])
    clean_b = esc(clean_text[i0:clean_end])
    clean_c = esc(clean_text[clean_end:clean_win_e])

    html = f"""
    <div style="background:#0b1220; padding:14px; border-radius:16px;">
      <div style="font-family:system-ui; font-weight:900; color:#e5e7eb; margin-bottom:10px;">
        {esc(title)}
      </div>

      <div style="font-family:ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;
                  color:#e5e7eb; line-height:1.6; font-size:14px;">
        <div style="color:#94a3b8; font-weight:800;">BEFORE (anonymized)</div>
        <div style="margin-top:6px;">
          <span style="color:#94a3b8;">{esc(raw_prefix)}</span>
          <span>{raw_a}</span>
          <span style="background:#fbbf24; color:#111827; font-weight:900; padding:1px 4px; border-radius:6px;">
            {raw_b if raw_b else "∅"}
          </span>
          <span>{raw_c}</span>
          <span style="color:#94a3b8;">{esc(raw_suffix)}</span>
        </div>

        <div style="margin:10px 0; text-align:center; color:#94a3b8; font-size:20px;">↓</div>

        <div style="color:#94a3b8; font-weight:800;">AFTER (synthetic)</div>
        <div style="margin-top:6px;">
          <span style="color:#94a3b8;">{esc(clean_prefix)}</span>
          <span>{clean_a}</span>
          <span style="background:#22c55e; color:#052e16; font-weight:900; padding:1px 4px; border-radius:6px;">
            {clean_b if clean_b else "∅"}
          </span>
          <span>{clean_c}</span>
          <span style="color:#94a3b8;">{esc(clean_suffix)}</span>
        </div>
      </div>
    </div>
    """
    display(HTML(html))


In [ ]:
i = 0
raw_text = df.iloc[i]["raw_text"]
clean_text = df.iloc[i]["clean_text"]

render_anonymized_to_synthetic_arrows(
    raw_text,
    clean_text,
    context_chars=35,
    title="Anonymized → Synthetic replacement (mini snippet)"
)
λλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλλ

In [ ]:
import re
from IPython.display import display, HTML

def find_at_tokens_spans(text):
    """
    Finds tokens starting with @ until whitespace.
    Returns spans: (s,e,"ANON")
    """
    spans = []
    for m in re.finditer(r"@\S+", text or ""):
        spans.append((m.start(), m.end(), "ANON"))
    return spans
def window_around_span(text, s, e, context_chars=60):
    n = len(text or "")
    win_s = max(0, s - context_chars)
    win_e = min(n, e + context_chars)
    return win_s, win_e
def render_before_after_by_spans(
    raw_text,
    clean_text,
    raw_focus_spans,          # list of (s,e,lab) on raw_text
    clean_focus_spans,        # list of (s,e,lab) on clean_text
    focus_idx=0,              # ποιο span να κεντράρουμε
    context_chars=60,
    title="Anonymized → Synthetic (mini snippet)"
):
    def esc(x):
        return (x.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;"))

    raw_text = raw_text or ""
    clean_text = clean_text or ""

    if not raw_focus_spans:
        display(HTML("<div>No anonymized tokens found in raw_text.</div>"))
        return

    focus_idx = min(max(0, focus_idx), len(raw_focus_spans)-1)
    fs, fe, _ = raw_focus_spans[focus_idx]

    raw_win_s, raw_win_e = window_around_span(raw_text, fs, fe, context_chars=context_chars)
    # στο clean βάζουμε ίδιο “μήκος window” περίπου, αλλά επειδή δεν ευθυγραμμίζονται τέλεια,
    # απλά παίρνουμε αρχή/τέλος με βάση το ίδιο context size γύρω από το πρώτο gold span αν υπάρχει,
    # αλλιώς παίρνουμε αρχή κειμένου.
    if clean_focus_spans:
        cs0, ce0, _ = clean_focus_spans[0]
        clean_win_s, clean_win_e = window_around_span(clean_text, cs0, ce0, context_chars=context_chars)
    else:
        clean_win_s, clean_win_e = 0, min(len(clean_text), 2*context_chars)

    # clip + shift spans into windows
    def clip_shift(spans, win_s, win_e):
        out = []
        for s,e,l in spans or []:
            s2, e2 = max(s, win_s), min(e, win_e)
            if s2 < e2:
                out.append((s2-win_s, e2-win_s, l))
        return out

    raw_snip = raw_text[raw_win_s:raw_win_e]
    clean_snip = clean_text[clean_win_s:clean_win_e]

    raw_sp = clip_shift(raw_focus_spans, raw_win_s, raw_win_e)
    clean_sp = clip_shift(clean_focus_spans, clean_win_s, clean_win_e)

    # simple renderer (background highlight)
    def render_line(snippet, spans, color, label):
        spans = sorted(spans, key=lambda x: x[0])
        html = ""
        i = 0
        for s,e,_ in spans:
            html += esc(snippet[i:s])
            html += f'<span style="background:{color}; color:#111827; font-weight:900; padding:1px 4px; border-radius:6px;">{esc(snippet[s:e])}</span>'
            i = e
        html += esc(snippet[i:])
        return f"""
        <div style="color:#94a3b8; font-weight:800; margin-top:6px;">{label}</div>
        <div style="margin-top:6px; white-space:pre-wrap;">{html}</div>
        """

    html = f"""
    <div style="background:#0b1220; padding:14px; border-radius:16px;">
      <div style="font-family:system-ui; font-weight:900; color:#e5e7eb; margin-bottom:10px;">
        {esc(title)}
      </div>

      <div style="font-family:ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;
                  color:#e5e7eb; line-height:1.6; font-size:14px;">
        {render_line(raw_snip, raw_sp, "#fbbf24", "BEFORE (anonymized tokens @...)")}
        <div style="margin:10px 0; text-align:center; color:#94a3b8; font-size:20px;">↓</div>
        {render_line(clean_snip, clean_sp, "#22c55e", "AFTER (synthetic entities)")}
      </div>
    </div>
    """
    display(HTML(html))


In [ ]:
i = 0
row = df.iloc[i]

raw_text = row["raw_text"]
clean_text = row["clean_text"]

raw_focus = find_at_tokens_spans(raw_text)                 # highlight @tokens
clean_focus = row.get("gold_spans_free_text", [])          # highlight synthetic entities (green)

render_before_after_by_spans(
    raw_text=raw_text,
    clean_text=clean_text,
    raw_focus_spans=raw_focus,
    clean_focus_spans=clean_focus,
    focus_idx=0,
    context_chars=70,
    title="Anonymized → Synthetic (mini snippet)"
)


In [ ]:
def fn_severity_stats(gold_spans, pred_spans):
    """
    Returns:
      fn_count,
      avg_fn_severity,
      weighted_fn_severity
    """
    gold = [(s, e) for s, e, _ in (gold_spans or [])]
    pred = [(s, e) for s, e, _ in (pred_spans or [])]

    fn_severities = []
    weighted_num = 0
    weighted_den = 0

    for g in gold:
        glen = max(0, g[1] - g[0])
        if glen == 0:
            continue

        covered = _union_covered_len(g, pred)
        coverage = covered / glen

        if coverage < 1.0:  # FN (partial or full)
            sev = 1 - coverage
            fn_severities.append(sev)

            weighted_num += (glen - covered)
            weighted_den += glen

    fn_count = len(fn_severities)
    avg_sev = sum(fn_severities) / fn_count if fn_count else 0.0
    weighted_sev = weighted_num / weighted_den if weighted_den else 0.0

    return fn_count, avg_sev, weighted_sev


In [ ]:
import pandas as pd

def fn_severity_summary(df, gold_col, pred_col):
    tot_fn = 0
    all_sev = []
    weighted_num = 0
    weighted_den = 0

    for _, r in df.iterrows():
        fn_count, avg_sev, weighted_sev = fn_severity_stats(
            r.get(gold_col),
            r.get(pred_col)
        )

        if fn_count > 0:
            tot_fn += fn_count
            # collect individual severities
            gold = [(s, e) for s, e, _ in (r.get(gold_col) or [])]
            pred = [(s, e) for s, e, _ in (r.get(pred_col) or [])]

            for g in gold:
                glen = g[1] - g[0]
                if glen <= 0:
                    continue
                covered = _union_covered_len(g, pred)
                coverage = covered / glen
                if coverage < 1.0:
                    sev = 1 - coverage
                    all_sev.append(sev)
                    weighted_num += (glen - covered)
                    weighted_den += glen

    avg_fn_sev = sum(all_sev) / len(all_sev) if all_sev else 0.0
    weighted_fn_sev = weighted_num / weighted_den if weighted_den else 0.0

    return {
        "FN count": tot_fn,
        "Avg FN severity": avg_fn_sev,
        "Weighted FN severity": weighted_fn_sev,
    }


In [ ]:
def build_fn_summary_table(df, gold_col, model_cols):
    rows = []

    for model_name, pred_col in model_cols.items():
        stats = fn_severity_summary(df, gold_col, pred_col)
        rows.append({
            "Model": model_name,
            "FN count": stats["FN count"],
            "Avg FN severity": round(stats["Avg FN severity"], 3),
            "Weighted FN severity": round(stats["Weighted FN severity"], 3),
        })

    return pd.DataFrame(rows)


In [ ]:
MODELS = {
    "spaCy-el": "free_pred_spans__spacy_el",
    "XLM-R": "free_pred_spans__xlm",
    "GR-BERT": "free_pred_spans__bert_gr",
    "LLaMA3": "free_pred_llama",
}

fn_summary_df = build_fn_summary_table(
    df,
    gold_col="gold_spans_free_text",
    model_cols=MODELS
)

fn_summary_df


,Model,FN count,Avg FN severity,Weighted FN severity
0,spaCy-el,27,0.575,0.565
1,XLM-R,24,0.097,0.091
2,GR-BERT,24,0.469,0.357
3,LLaMA3,16,0.476,0.303
